# Evaluate scDRS-FM simulation results

This notebook evaluates all simulation outputs generated for the project:

1. **Original simulations**: varying the number of causal populations (`src_n`).
2. **Cell-percent simulations**: fixed at 3 causal populations, varying the minimum cell percentage required for each causal population.
3. **Causal-gene simulations**: fixed at 3 causal populations, varying causal genes per population (`25, 50, 75, 100`).
4. **MAGIC vs KNN denoising comparison**: original simulations evaluated with the default MAGIC denoising and the KNN denoising run.

The analyses are configuration-driven: each simulation family or comparison is defined once, then the shared evaluators and plotting functions are reused for standard metrics, independent-signal summaries, and independent-signal precision/recall.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple
from urllib.parse import unquote
import ast
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
from scipy import stats
from sklearn.metrics import auc, precision_recall_curve
from statsmodels.stats.multitest import multipletests


## Global configuration

Edit paths and plotting thresholds here. The defaults match the simulation-generation and Slurm scripts from this project.


In [2]:
# === scDRS-FM reproduction: portable path anchor (injected, P5) ===
import os as _os
from pathlib import Path as _Path
def _find_repo_root():
    # 1) explicit override wins
    env = _os.environ.get('SCDRSFM_BASE')
    if env:
        return _Path(env)
    # 2) search upward from CWD for the reproduction repo root
    #    (a directory containing both 'scDRS-FM-main' and 'scripts')
    here = _Path.cwd().resolve()
    for d in (here, *here.parents):
        if (d / 'scDRS-FM-main').is_dir() and (d / 'scripts').is_dir():
            return d
    # 3) last resort: current working directory
    return here
BASE = _find_repo_root()
DATA = BASE / 'data'
RESULTS = BASE / 'results'
MAGMA_REF = BASE / 'magma_ref'
assert BASE.exists(), f'Repro root not found (set SCDRSFM_BASE to the repo root): {BASE}'


In [3]:
# -----------------------------------------------------------------------------
# Paths and dataset-level settings
# -----------------------------------------------------------------------------
BASE_DIR = RESULTS / "sim" / "simulation_data"
DATASET_PREFIX = "TMS_FACS"
LABEL_KEY = "leiden"
H5AD_PATH = BASE_DIR / "adata" / f"{DATASET_PREFIX}_10k_DE.h5ad"

# Statistical thresholds
ALPHA = 0.10
FDR_ALPHA = 0.10
MIN_FRAC_CELLS_MARG_SIG = 0.05
DISCOVERY_MIN_FRAC_CELLLEVEL = 0.05
MATCH_MIN_FRAC_OF_CAUSAL = 0.05

# scDRS Leiden baseline for independent-signal precision/recall.
SCDRS_LEIDEN_RESOLUTION = 1.0
SCDRS_LEIDEN_KEY = "scdrs_leiden_res1p0"
SCDRS_LEIDEN_MIN_FRAC_SIG = 0.05

# Input p-value columns.
PVAL_COL_CANDIDATES = ("pval",)
MARG_PVAL_COL_CANDIDATES = ("pval",)
COND_PVAL_COL_CANDIDATES = ("pval",)

# Conditional-like methods to evaluate in the standard FDR / Power / AUPRC plots.
COND_TESTS_SCDRSPLUS = [
    ("conditional", "conditional", ("pval",)),
    ("conditional_susie", "conditional_susie", ("pval",)),
    ("conditional_ridge_focal", "conditional_ridge_focal", ("pval",)),
    ("conditional_wls", "conditional_wls", ("pval",)),
]
COND_TESTS_SCDRS = [("conditional", ("pval",))]

# Figure labels. Internal method names can keep using the original code strings;
# all displayed text uses scDRS-FM for the new method name.
METHOD_NAME_MAP = {
    "scdrs+_marginal": "scDRS-FM marginal",
    "scdrs+_conditional": "scDRS-FM",
    "scdrs+_conditional_susie": "scDRS-FM w/ SuSiE",
    "scdrs+_conditional_ridge_focal": "scDRS-FM w/ penalizing focal metacell",
    "scdrs+_conditional_wls": "scDRS-FM w/ WLS",
    "scdrs_marginal": "scDRS",
    "scdrs_marginal_leiden": "scDRS w/ Leiden",
    "scdrs_conditional": "scDRS-FM no-denoise",
    "magic": "MAGIC",
    "knn": "KNN",
}
METHOD_ORDER = [
    "scdrs+_conditional",
    #"scdrs+_conditional_susie",
    #"scdrs+_conditional_ridge_focal",
    #"scdrs+_conditional_wls",
    #"scdrs_marginal",
    "scdrs+_marginal",
    "scdrs_conditional",
]
METRICS = [("fdr", "FDP"), ("power", "Power"), ("auprc", "AUPRC")]

# Independent-signal columns. When both columns are present, independent_signal_multi
# is treated as the default for all independent-signal analyses.
PRIMARY_SIGNAL_COL_CANDIDATES = (
    "independent_signal_multi",
    "independent_signal",
)
SIGNAL_COLS = [
    "independent_signal_multi",
    "independent_signal",
    "independent_signal_wls",
    "independent_signal_factor",
    "independent_signal_nmf",
    "independent_signal_lda",
    "independent_signal_direction",
]
SIGNAL_DISPLAY_NAMES = {
    "independent_signal_multi": "scDRS-FM",
    "independent_signal": "scDRS-FM",
    "independent_signal_wls": "scDRS-FM w/ WLS",
    "independent_signal_factor": "scDRS-FM w/ Factor analysis",
    "independent_signal_nmf": "scDRS-FM w/ NMF",
    "independent_signal_lda": "scDRS-FM w/ LDA",
    "independent_signal_direction": "scDRS-FM w/ Direction-aware",
    "scdrs_marginal": "scDRS (all-in-one)",
    "scdrs_marginal_leiden": "scDRS w/ Leiden",
}

# Set to True to drop files whose primary independent-signal column contains -1.
EXCLUDE_FILES_WITH_NEG1_INDEP_SIGNAL_FROM_PLOTS = False


## Simulation-set definitions

Each simulation set specifies its geneset directory, prediction table, result directories, filename parser, merge keys, and x-axis variables. The KNN denoising run is defined as an auxiliary set and compared through the same method-comparison machinery used elsewhere.


In [4]:
@dataclass(frozen=True)
class SimulationSetConfig:
    key: str
    title: str
    geneset_dir: Path
    predictions_csv: Path
    scdrsplus_root: Path
    scdrs_root: Path
    filename_kind: str
    geneset_glob: str
    prediction_merge_keys: Tuple[str, ...]
    core_x_col: str
    core_x_label: str
    indep_x_col: str
    indep_x_label: str
    output_suffix: str
    # Optional recursive roots for independent-signal files. The first existing
    # root is used. If none exist, scdrsplus_root is used.
    indep_root_candidates: Tuple[Path, ...] = ()


@dataclass(frozen=True)
class MethodComparisonMember:
    cfg_key: str
    source_method: str
    plot_method: str
    label: str


@dataclass(frozen=True)
class MethodComparisonConfig:
    key: str
    title: str
    base_cfg_key: str
    output_suffix: str
    members: Tuple[MethodComparisonMember, ...]

    @property
    def method_order(self) -> list[str]:
        return [member.plot_method for member in self.members]

    @property
    def method_name_map(self) -> dict[str, str]:
        return {member.plot_method: member.label for member in self.members}


SIMULATION_SETS: list[SimulationSetConfig] = [
    SimulationSetConfig(
        key="original",
        title="Original simulations",
        geneset_dir=BASE_DIR / "gs_de_overlap",
        predictions_csv=BASE_DIR / "predictions.csv",
        scdrsplus_root=BASE_DIR / "scdrs+_results",
        scdrs_root=BASE_DIR / "scdrs_results",
        filename_kind="original",
        geneset_glob=f"{DATASET_PREFIX}_*_*_*_src*.gs",
        prediction_merge_keys=("cluster", "replicate", "src_n"),
        core_x_col="src_n",
        core_x_label="Number of causal populations",
        indep_x_col="causal_clusters",
        indep_x_label="Causal clusters",
        output_suffix="original",
        indep_root_candidates=(BASE_DIR / "scdrs+_results" / "indep", BASE_DIR / "scdrs+_results"),
    ),
    SimulationSetConfig(
        key="sims_percent",
        title="Cell-percent simulations",
        geneset_dir=BASE_DIR / "gs_cell_pct",
        predictions_csv=BASE_DIR / "predictions_cell_pct.csv",
        scdrsplus_root=BASE_DIR / "scdrs+_results_cell_pct",
        scdrs_root=BASE_DIR / "scdrs_results_cell_pct",
        filename_kind="cell_pct",
        geneset_glob=f"{DATASET_PREFIX}_*_*_pct*_ov*_src*.gs",
        prediction_merge_keys=("cluster", "replicate", "src_n", "min_cell_percent"),
        core_x_col="min_cell_percent",
        core_x_label="Minimum cells per causal cluster (%)",
        indep_x_col="min_cell_percent",
        indep_x_label="Minimum cells per causal cluster (%)",
        output_suffix="sims_percent",
        indep_root_candidates=(
            BASE_DIR / "scdrs+_results_cell_pct" / "indep",
            BASE_DIR / "scdrs+_results_cell_pct",
            BASE_DIR / "sims_percent",
        ),
    ),
    SimulationSetConfig(
        key="sims_genes",
        title="Causal-gene simulations",
        geneset_dir=BASE_DIR / "gs_causal_genes",
        predictions_csv=BASE_DIR / "predictions_causal_genes.csv",
        scdrsplus_root=BASE_DIR / "scdrs+_results_causal_genes",
        scdrs_root=BASE_DIR / "scdrs_results_causal_genes",
        filename_kind="causal_genes",
        geneset_glob=f"{DATASET_PREFIX}_*_*_ov*_src*.gs",
        prediction_merge_keys=("cluster", "replicate", "src_n", "overlap_k"),
        core_x_col="overlap_k",
        core_x_label="Causal genes per cluster",
        indep_x_col="overlap_k",
        indep_x_label="Causal genes per cluster",
        output_suffix="sims_genes",
        indep_root_candidates=(
            BASE_DIR / "scdrs+_results_causal_genes" / "indep",
            BASE_DIR / "scdrs+_results_causal_genes",
            BASE_DIR / "sims_genes",
        ),
    ),
]

# Auxiliary evaluation set used by the denoising comparison. It is evaluated by
# the same functions as the main simulation sets but is not plotted as a separate
# simulation family.
KNN_ORIGINAL_CFG = replace(
    SIMULATION_SETS[0],
    key="original_knn",
    title="Original simulations — KNN denoising",
    scdrsplus_root=BASE_DIR / "scdrs+_results_knn",
    output_suffix="original_knn",
    indep_root_candidates=(
        BASE_DIR / "scdrs+_results_knn" / "indep",
        BASE_DIR / "scdrs+_results_knn",
    ),
)

AUXILIARY_SIMULATION_SETS: list[SimulationSetConfig] = [KNN_ORIGINAL_CFG]
ALL_EVALUATION_SETS: list[SimulationSetConfig] = SIMULATION_SETS + AUXILIARY_SIMULATION_SETS
SIM_BY_KEY = {cfg.key: cfg for cfg in ALL_EVALUATION_SETS}

DENOISING_COMPARISON = MethodComparisonConfig(
    key="magic_vs_knn",
    title="Denoising comparison: MAGIC vs KNN",
    base_cfg_key="original",
    output_suffix="magic_vs_knn",
    members=(
        MethodComparisonMember(
            cfg_key="original",
            source_method="scdrs+_conditional",
            plot_method="magic",
            label="MAGIC",
        ),
        MethodComparisonMember(
            cfg_key="original_knn",
            source_method="scdrs+_conditional",
            plot_method="knn",
            label="KNN",
        ),
    ),
)

METHOD_COMPARISONS: list[MethodComparisonConfig] = [DENOISING_COMPARISON]


## Shared utilities


In [5]:
def _pick_first_existing_col(df: pd.DataFrame, candidates: Sequence[str], *, what: str) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"{what}: none of these columns exist: {tuple(candidates)}")


def bh_fdr_mask(pvals: np.ndarray, alpha: float) -> np.ndarray:
    p = np.asarray(pvals, dtype=np.float64)
    ok = np.isfinite(p)
    out = np.zeros_like(ok, dtype=bool)
    if ok.sum() == 0:
        return out
    rej, _, _, _ = multipletests(p[ok], alpha=alpha, method="fdr_bh")
    out[ok] = rej
    return out


def _read_adata_obs_cols(h5ad_path: Path, keys: Sequence[str]) -> tuple[pd.Index, dict[str, np.ndarray]]:
    """Read only obs names and requested obs columns from an h5ad file."""
    ad = sc.read_h5ad(str(h5ad_path), backed="r")
    obs_names = ad.obs_names.astype(str)

    out: dict[str, np.ndarray] = {}
    for key in keys:
        if key not in ad.obs:
            raise KeyError(f"{key!r} not in adata.obs for {h5ad_path}")
        out[key] = ad.obs[key].astype(str).to_numpy()

    try:
        ad.file.close()
    except Exception:
        try:
            ad._file.close()
        except Exception:
            pass

    return obs_names, out


def _read_trait_from_gs(gs_path: Path) -> str:
    with open(gs_path, "r", encoding="utf-8") as handle:
        _ = handle.readline()
        line = handle.readline()
    if not line:
        raise ValueError(f"Empty geneset file: {gs_path}")
    trait = line.split("\t", 1)[0].strip()
    if not trait:
        raise ValueError(f"Could not parse TRAIT from {gs_path}")
    return trait


def _parse_trait_info(trait: str) -> dict[str, Any]:
    """Parse traits produced by the simulation-generation notebook.

    Expected segments look like:
      target__rep0__ov50__src3__donors0+4+5

    Extra segments, such as cellpct1, are preserved in `extras` and ignored by
    the core donor parsing.
    """
    parts = trait.split("__")
    if not parts or not parts[0]:
        raise ValueError(f"Malformed trait: {trait!r}")

    target = unquote(parts[0])
    rep: int | None = None
    ov: int | None = None
    src_n: int | None = None
    donors: list[str] | None = None
    extras: list[str] = []

    for seg in parts[1:]:
        if seg.startswith("rep"):
            rep = int(seg[3:])
        elif seg.startswith("ov"):
            ov = int(seg[2:])
        elif seg.startswith("src"):
            src_n = int(seg[3:])
        elif seg.startswith("donors"):
            enc = seg[len("donors"):]
            donors = [unquote(tok) for tok in enc.split("+")] if enc else []
        else:
            extras.append(seg)

    if src_n is None:
        src_n = 1
    if donors is None or len(donors) == 0:
        donors = [target]

    return {
        "target": str(target),
        "replicate": rep,
        "overlap_k": ov,
        "src_n": int(src_n),
        "donors": [str(d) for d in donors],
        "extras": extras,
    }


def _parse_src_token(token: str) -> int:
    if not token.startswith("src"):
        raise ValueError(f"Unexpected source token: {token}")
    return int(token[len("src"):])


def parse_geneset_filename(gs_path: Path, cfg: SimulationSetConfig) -> dict[str, Any]:
    """Parse metadata encoded in a geneset filename."""
    stem = gs_path.stem
    prefix = f"{DATASET_PREFIX}_"
    if not stem.startswith(prefix):
        raise ValueError(f"Unexpected geneset prefix: {gs_path.name}")

    parts = stem[len(prefix):].split("_")

    if cfg.filename_kind == "original":
        if len(parts) != 4:
            raise ValueError(f"Expected 4 filename fields for original set: {gs_path.name}")
        cluster, rep, overlap_k, src = parts
        return {
            "cluster": int(cluster),
            "replicate": int(rep),
            "overlap_k": int(overlap_k),
            "src_n": _parse_src_token(src),
        }

    if cfg.filename_kind == "cell_pct":
        if len(parts) != 5:
            raise ValueError(f"Expected 5 filename fields for cell-percent set: {gs_path.name}")
        cluster, rep, pct, ov, src = parts
        if not pct.startswith("pct") or not ov.startswith("ov"):
            raise ValueError(f"Malformed cell-percent filename: {gs_path.name}")
        return {
            "cluster": int(cluster),
            "replicate": int(rep),
            "min_cell_percent": int(pct[len("pct"):]),
            "overlap_k": int(ov[len("ov"):]),
            "src_n": _parse_src_token(src),
        }

    if cfg.filename_kind == "causal_genes":
        if len(parts) != 4:
            raise ValueError(f"Expected 4 filename fields for causal-gene set: {gs_path.name}")
        cluster, rep, ov, src = parts
        if not ov.startswith("ov"):
            raise ValueError(f"Malformed causal-gene filename: {gs_path.name}")
        return {
            "cluster": int(cluster),
            "replicate": int(rep),
            "overlap_k": int(ov[len("ov"):]),
            "src_n": _parse_src_token(src),
        }

    raise ValueError(f"Unknown filename kind: {cfg.filename_kind}")


def result_dir_for_run(root: Path, cfg: SimulationSetConfig, meta: dict[str, Any]) -> Path:
    """Return the result directory used by the Slurm scripts for one run."""
    base = root / str(int(meta["cluster"])) / str(int(meta["replicate"]))
    if cfg.filename_kind == "cell_pct":
        base = base / f"pct{int(meta['min_cell_percent'])}"
    elif cfg.filename_kind == "causal_genes":
        base = base / f"ov{int(meta['overlap_k'])}"
    return base / f"src{int(meta['src_n'])}"


def geneset_files(cfg: SimulationSetConfig) -> list[Path]:
    if not cfg.geneset_dir.exists():
        raise FileNotFoundError(cfg.geneset_dir)
    files = sorted(cfg.geneset_dir.glob(cfg.geneset_glob))
    if not files:
        raise FileNotFoundError(f"No genesets matched {cfg.geneset_glob!r} in {cfg.geneset_dir}")
    return files


def _parse_label_set(x: Any) -> set[str]:
    if x is None:
        return set()
    if isinstance(x, (list, tuple, set)):
        return {str(v) for v in x if v is not None and str(v).lower() not in ("", "nan")}

    s = str(x).strip()
    if s == "" or s.lower() == "nan":
        return set()

    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")) or (s.startswith("{") and s.endswith("}")):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, (list, tuple, set)):
                return {str(t) for t in v if t is not None and str(t).lower() not in ("", "nan")}
        except Exception:
            pass

    for delim in [";", ",", "+", "|"]:
        if delim in s:
            return {tok.strip() for tok in s.split(delim) if tok.strip() and tok.strip().lower() != "nan"}
    return {s}


def load_predictions(cfg: SimulationSetConfig) -> pd.DataFrame:
    if not cfg.predictions_csv.exists():
        raise FileNotFoundError(cfg.predictions_csv)
    pred = pd.read_csv(cfg.predictions_csv)
    for col in ("cluster", "replicate", "src_n", "overlap_k", "min_cell_percent", "causal_genes_per_cluster"):
        if col in pred.columns:
            pred[col] = pd.to_numeric(pred[col], errors="coerce")
    return pred


def choose_existing_root(candidates: Sequence[Path], fallback: Path) -> Path:
    for root in candidates:
        if root.exists():
            return root
    return fallback



def choose_default_signal_col(available_cols: Iterable[str]) -> Optional[str]:
    """Choose the default independent-signal column for a file or result table."""
    available = {str(col) for col in available_cols}
    for col in PRIMARY_SIGNAL_COL_CANDIDATES:
        if col in available:
            return col
    return None


def signal_display_name(signal_col: str, default_signal_col: Optional[str] = None) -> str:

    return SIGNAL_DISPLAY_NAMES.get(str(signal_col), str(signal_col))


def ordered_signal_cols(cols: Iterable[str]) -> list[str]:
    cols_set = {str(col) for col in cols}
    ordered = [col for col in SIGNAL_COLS if col in cols_set]
    ordered.extend(sorted(cols_set - set(ordered)))
    return ordered


def compute_or_load_scdrs_leiden_labels(
    h5ad_path: Path,
    *,
    obs_names: pd.Index,
    resolution: float = SCDRS_LEIDEN_RESOLUTION,
    key_added: str = SCDRS_LEIDEN_KEY,
) -> pd.Series:
    """Compute Leiden clusters once for the scDRS cell-level baseline.

    These labels are deliberately separate from the simulation truth labels in
    `LABEL_KEY`. They are used only to split scDRS marginally associated cells
    into candidate independent signals.
    """
    adata = sc.read_h5ad(str(h5ad_path))

    if key_added not in adata.obs:
        if "neighbors" not in adata.uns:
            if "X_pca" not in adata.obsm:
                n_comps = int(max(2, min(50, adata.n_obs - 1, adata.n_vars - 1)))
                sc.pp.pca(adata, n_comps=n_comps)
            sc.pp.neighbors(adata, use_rep="X_pca" if "X_pca" in adata.obsm else None)
        sc.tl.leiden(adata, resolution=float(resolution), key_added=key_added)

    labels = adata.obs[key_added].astype(str).copy()
    labels.index = labels.index.astype(str)
    return labels.reindex(obs_names.astype(str))


## Standard scDRS/scDRS-FM metric evaluation

This section reproduces the original FDR/Power/AUPRC analysis and applies the same evaluator to the two additional simulation sets and the KNN denoising run.


In [6]:
def _find_first_existing(out_dir: Path, candidates: Sequence[str]) -> Optional[Path]:
    for name in candidates:
        path = out_dir / name
        if path.exists():
            return path
    return None


def _find_marginal_file(out_dir: Path, prefixes: Sequence[str]) -> Optional[Path]:
    candidates: list[str] = []
    for pref in prefixes:
        candidates.extend([f"{pref}.marginal_score.gz", f"{pref}.marginal.score.gz", f"{pref}.score.gz"])
    return _find_first_existing(out_dir, candidates)


def _find_conditional_file(out_dir: Path, prefixes: Sequence[str], conditional_tag: str = "conditional") -> Optional[Path]:
    candidates: list[str] = []
    for pref in prefixes:
        if conditional_tag == "conditional":
            candidates.extend([
                f"{pref}.conditional.tagging_score.gz",
                f"{pref}.tagging_score.gz",
                f"{pref}.conditional_score.gz",
            ])
        else:
            candidates.extend([f"{pref}.{conditional_tag}.tagging_score.gz", f"{pref}.{conditional_tag}_score.gz"])
    return _find_first_existing(out_dir, candidates)


def _read_marginal(marg_file: Path, obs_names: pd.Index, alpha: float) -> tuple[pd.Index, np.ndarray]:
    df_marg = pd.read_csv(marg_file, sep="\t", compression="gzip", index_col=0)
    pcol = _pick_first_existing_col(df_marg, PVAL_COL_CANDIDATES, what="marginal p-value")
    p_marg_vec = df_marg[pcol].reindex(obs_names).fillna(1.0).to_numpy(dtype=np.float64)

    rej = bh_fdr_mask(df_marg[pcol].to_numpy(dtype=np.float64), alpha)
    sig_cells = df_marg.index[rej].astype(str)
    return obs_names.intersection(sig_cells), p_marg_vec


def _normalize_metacell_index_to_int(df: pd.DataFrame) -> pd.DataFrame:
    mc_ids_num = pd.to_numeric(pd.Index(df.index), errors="coerce")
    out = df.copy()
    out.index = mc_ids_num
    out = out.loc[out.index.notna()]
    out.index = out.index.astype(int)
    return out


def _read_conditional_metacell_with_pcol(
    cond_file: Path,
    obs_names: pd.Index,
    alpha: float,
    pcol_candidates: Sequence[str],
) -> tuple[pd.Index, np.ndarray]:
    """Return cells in significant metacells and a per-cell conditional p-value vector."""
    df_cond = pd.read_csv(cond_file, sep="\t", compression="gzip", index_col=0)
    pcol = _pick_first_existing_col(df_cond, pcol_candidates, what="conditional-like p-value")
    df_cond = _normalize_metacell_index_to_int(df_cond)

    rej = bh_fdr_mask(df_cond[pcol].to_numpy(dtype=np.float64), alpha)
    sig_metacells = set(df_cond.index.to_numpy()[rej].astype(int, copy=False))

    if sig_metacells:
        all_cells: list[str] = []
        for cell_ids in df_cond.loc[sorted(sig_metacells), "cell_ids"].astype(str).tolist():
            if cell_ids and cell_ids != "nan":
                all_cells.extend([x for x in cell_ids.split(",") if x])
        cells_in_sig_metacells = obs_names.intersection(pd.Index(all_cells, dtype=str))
    else:
        cells_in_sig_metacells = pd.Index([], dtype=str)

    tmp = df_cond[["cell_ids", pcol]].copy()
    tmp["cell_id"] = tmp["cell_ids"].astype(str).str.split(",")
    tmp = tmp.explode("cell_id")
    tmp["cell_id"] = tmp["cell_id"].astype(str)
    tmp = tmp[(tmp["cell_id"].notna()) & (tmp["cell_id"] != "") & (tmp["cell_id"] != "nan")]

    cell_to_p = tmp.groupby("cell_id")[pcol].min()
    p_cond_vec = cell_to_p.reindex(obs_names).fillna(1.0).to_numpy(dtype=np.float64)
    return cells_in_sig_metacells, p_cond_vec


def _metrics_from_sig_and_pvals(
    *,
    obs_names: pd.Index,
    causal_mask: np.ndarray,
    sig_cells: pd.Index,
    pvals_vec: np.ndarray,
) -> dict[str, float]:
    causal_mask = np.asarray(causal_mask, dtype=bool)
    causal_cells = set(obs_names[causal_mask].astype(str).tolist())
    noncausal_cells = set(obs_names[~causal_mask].astype(str).tolist())
    sig_set = set(sig_cells.astype(str).tolist())

    fp = len(sig_set & noncausal_cells)
    tp = len(sig_set & causal_cells)
    n_sig = len(sig_set)
    n_causal = int(causal_mask.sum())

    fdr = fp / max(n_sig, 1)
    power = tp / max(n_causal, 1)

    true_labels = causal_mask.astype(int)
    scores = 1.0 - np.asarray(pvals_vec, dtype=np.float64)
    if true_labels.sum() == 0:
        auprc = np.nan
    else:
        precision, recall, _ = precision_recall_curve(true_labels, scores)
        auprc = auc(recall, precision)

    return {"n_sig": float(n_sig), "fdr": float(fdr), "power": float(power), "auprc": float(auprc)}


def evaluate_marginal_only(
    *,
    obs_names: pd.Index,
    causal_mask: np.ndarray,
    out_dir: Path,
    prefixes: Sequence[str],
    alpha: float,
) -> dict[str, float]:
    marg_path = _find_marginal_file(out_dir, prefixes)
    if marg_path is None:
        return {"n_sig": np.nan, "fdr": np.nan, "power": np.nan, "auprc": np.nan}

    sig_marg, p_marg_vec = _read_marginal(marg_path, obs_names, alpha)
    return _metrics_from_sig_and_pvals(
        obs_names=obs_names,
        causal_mask=causal_mask,
        sig_cells=sig_marg,
        pvals_vec=p_marg_vec,
    )


def evaluate_conditional_like(
    *,
    obs_names: pd.Index,
    causal_mask: np.ndarray,
    out_dir: Path,
    prefixes: Sequence[str],
    alpha: float,
    pcol_candidates: Sequence[str],
    conditional_tag: str = "conditional",
) -> dict[str, float]:
    marg_path = _find_marginal_file(out_dir, prefixes)
    if marg_path is None:
        return {"n_sig": np.nan, "fdr": np.nan, "power": np.nan, "auprc": np.nan}

    sig_marg, p_marg_vec = _read_marginal(marg_path, obs_names, alpha)

    cond_path = _find_conditional_file(out_dir, prefixes, conditional_tag=conditional_tag)
    if cond_path is None:
        return {"n_sig": np.nan, "fdr": np.nan, "power": np.nan, "auprc": np.nan}

    try:
        cells_in_sig_metacells, p_cond_vec = _read_conditional_metacell_with_pcol(
            cond_path,
            obs_names,
            alpha,
            pcol_candidates,
        )
    except Exception:
        return {"n_sig": np.nan, "fdr": np.nan, "power": np.nan, "auprc": np.nan}

    sig_intersection = sig_marg.intersection(cells_in_sig_metacells)
    p_combined = np.maximum(p_marg_vec, p_cond_vec)
    return _metrics_from_sig_and_pvals(
        obs_names=obs_names,
        causal_mask=causal_mask,
        sig_cells=sig_intersection,
        pvals_vec=p_combined,
    )


def evaluate_core_simulation_set(
    cfg: SimulationSetConfig,
    *,
    obs_names: pd.Index,
    labels: np.ndarray,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    skipped = 0

    for gs_path in geneset_files(cfg):
        try:
            meta = parse_geneset_filename(gs_path, cfg)
            trait = _read_trait_from_gs(gs_path)
            trait_info = _parse_trait_info(trait)
        except Exception:
            skipped += 1
            continue

        donors = [str(d) for d in trait_info["donors"]]
        causal_mask = np.isin(labels.astype(str), donors)
        prefixes = [trait, gs_path.name, gs_path.stem]

        scdrsplus_out = result_dir_for_run(cfg.scdrsplus_root, cfg, meta)
        scdrs_out = result_dir_for_run(cfg.scdrs_root, cfg, meta)

        donors_other = [d for d in donors if str(d) != str(trait_info["target"])]
        base_row: dict[str, Any] = {
            "simulation_set": cfg.key,
            "cluster": int(meta["cluster"]),
            "replicate": int(meta["replicate"]),
            "src_n": int(meta["src_n"]),
            "overlap_k": int(meta["overlap_k"]),
            "causal_genes_per_cluster": int(meta["overlap_k"]),
            "target_leiden": str(trait_info["target"]),
            "target_leiden_from_filename": str(meta["cluster"]),
            "trait_src_n_clusters": int(trait_info["src_n"]),
            "trait_donor_clusters": ";".join(donors),
            "trait_donor_clusters_n": len(donors),
            "trait_donor_clusters_other": ";".join(donors_other),
            "trait_donor_clusters_other_n": len(donors_other),
            "geneset_file": gs_path.name,
            "geneset_path": str(gs_path),
        }
        if "min_cell_percent" in meta:
            base_row["min_cell_percent"] = int(meta["min_cell_percent"])

        # scDRS+ marginal and conditional-like outputs.
        res = evaluate_marginal_only(
            obs_names=obs_names,
            causal_mask=causal_mask,
            out_dir=scdrsplus_out,
            prefixes=prefixes,
            alpha=ALPHA,
        )
        rows.append({**base_row, "method": "scdrs+_marginal", **res})

        for suffix, conditional_tag, pcol_cands in COND_TESTS_SCDRSPLUS:
            res = evaluate_conditional_like(
                obs_names=obs_names,
                causal_mask=causal_mask,
                out_dir=scdrsplus_out,
                prefixes=prefixes,
                alpha=ALPHA,
                pcol_candidates=pcol_cands,
                conditional_tag=conditional_tag,
            )
            rows.append({**base_row, "method": f"scdrs+_{suffix}", **res})

        # scDRS baseline outputs.
        res = evaluate_marginal_only(
            obs_names=obs_names,
            causal_mask=causal_mask,
            out_dir=scdrs_out,
            prefixes=prefixes,
            alpha=ALPHA,
        )
        rows.append({**base_row, "method": "scdrs_marginal", **res})

        for suffix, pcol_cands in COND_TESTS_SCDRS:
            res = evaluate_conditional_like(
                obs_names=obs_names,
                causal_mask=causal_mask,
                out_dir=scdrs_out,
                prefixes=prefixes,
                alpha=ALPHA,
                pcol_candidates=pcol_cands,
            )
            rows.append({**base_row, "method": f"scdrs_{suffix}", **res})

    out = pd.DataFrame(rows)
    if out.empty:
        print(f"{cfg.title}: no evaluable genesets found.")
        return out

    sort_cols = [c for c in ["simulation_set", cfg.core_x_col, "cluster", "src_n", "replicate", "method"] if c in out.columns]
    out = out.sort_values(sort_cols).reset_index(drop=True)
    if skipped:
        print(f"{cfg.title}: skipped {skipped} geneset files that did not match expected parsing rules.")
    return out


In [7]:
# Load cell labels once and evaluate the standard metrics for every configured result set.
if not H5AD_PATH.exists():
    raise FileNotFoundError(H5AD_PATH)

obs_names, obs_cols = _read_adata_obs_cols(H5AD_PATH, (LABEL_KEY,))
obs_names = obs_names.astype(str)
labels = np.asarray(obs_cols[LABEL_KEY], dtype=str)

core_results: dict[str, pd.DataFrame] = {}
for cfg in ALL_EVALUATION_SETS:
    print(f"Evaluating standard metrics: {cfg.title}")
    core_results[cfg.key] = evaluate_core_simulation_set(cfg, obs_names=obs_names, labels=labels)

# Backward-compatible alias used by the original notebook.
results = core_results["original"]
results.head()


Evaluating standard metrics: Original simulations


Evaluating standard metrics: Cell-percent simulations


Evaluating standard metrics: Causal-gene simulations


Evaluating standard metrics: Original simulations — KNN denoising


,simulation_set,cluster,replicate,src_n,overlap_k,causal_genes_per_cluster,target_leiden,target_leiden_from_filename,trait_src_n_clusters,trait_donor_clusters,trait_donor_clusters_n,trait_donor_clusters_other,trait_donor_clusters_other_n,geneset_file,geneset_path,method,n_sig,fdr,power,auprc
0,original,0,0,1,50,50,0,0,1,0,1,,0,TMS_FACS_0_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,930.0,0.064516,0.964523,0.962484
1,original,0,0,1,50,50,0,0,1,0,1,,0,TMS_FACS_0_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional_ridge_focal,NaN,NaN,NaN,NaN
2,original,0,0,1,50,50,0,0,1,0,1,,0,TMS_FACS_0_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional_susie,NaN,NaN,NaN,NaN
3,original,0,0,1,50,50,0,0,1,0,1,,0,TMS_FACS_0_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional_wls,NaN,NaN,NaN,NaN
4,original,0,0,1,50,50,0,0,1,0,1,,0,TMS_FACS_0_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_marginal,1394.0,0.352941,1.000000,0.984004


## Standard metric figures

The following figures use the same three-panel style for all simulation families, and the denoising comparison is produced by the same generic plotting functions.


In [8]:
CORE_PLOT_PARAMS = dict(
    figsize=(18, 8),
    dpi=300,
    suptitle_fontsize=36,
    axis_label_fontsize=32,
    tick_label_fontsize=20,
    legend_fontsize=24,
    err_linewidth=2.0,
    capsize=4,
    suptitle_y=0.84,
    legend_y=1.215,
    tight_rect=(0, 0.07, 1, 0.84),
)


def mean_ci95_if_n_ge_3(x: np.ndarray) -> tuple[float, float, float, int]:
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = int(x.size)
    if n == 0:
        return np.nan, np.nan, np.nan, 0
    m = float(np.mean(x))
    if n < 3:
        return m, m, m, n
    se = float(np.std(x, ddof=1) / np.sqrt(n))
    half = 1.96 * se
    return m, m - half, m + half, n


def merge_with_predictions_for_plot(df: pd.DataFrame, cfg: SimulationSetConfig) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    pred = load_predictions(cfg)
    keys = [key for key in cfg.prediction_merge_keys if key in df.columns and key in pred.columns]
    if not keys:
        return df.copy()

    left = df.copy()
    right = pred[keys].drop_duplicates().copy()
    for col in keys:
        left[col] = pd.to_numeric(left[col], errors="coerce")
        right[col] = pd.to_numeric(right[col], errors="coerce")
    return left.merge(right, on=keys, how="inner")


def summarize_core_metrics(
    df: pd.DataFrame,
    cfg: SimulationSetConfig,
    *,
    method_order: Optional[Sequence[str]] = None,
    method_name_map: Optional[dict[str, str]] = None,
) -> pd.DataFrame:
    plot_df = merge_with_predictions_for_plot(df, cfg)
    if plot_df.empty:
        return pd.DataFrame()

    label_map = {**METHOD_NAME_MAP, **(method_name_map or {})}
    order = list(method_order or METHOD_ORDER)

    required = {"method", cfg.core_x_col} | {metric for metric, _ in METRICS}
    missing = required - set(plot_df.columns)
    if missing:
        raise KeyError(f"{cfg.title}: results missing columns: {sorted(missing)}")

    plot_df[cfg.core_x_col] = pd.to_numeric(plot_df[cfg.core_x_col], errors="coerce")
    for metric, _ in METRICS:
        plot_df[metric] = pd.to_numeric(plot_df[metric], errors="coerce")

    rows: list[dict[str, Any]] = []
    for (method, xval), group in plot_df.groupby(["method", cfg.core_x_col], dropna=False):
        rec: dict[str, Any] = {
            "method": method,
            "method_label": label_map.get(method, method),
            cfg.core_x_col: xval,
            "n_points": int(group.shape[0]),
        }
        for metric, _ in METRICS:
            m, lo, hi, n = mean_ci95_if_n_ge_3(group[metric].to_numpy())
            rec[f"{metric}_mean"] = m
            rec[f"{metric}_lo"] = lo
            rec[f"{metric}_hi"] = hi
            rec[f"{metric}_n"] = n
        rows.append(rec)

    summ = pd.DataFrame(rows)
    if summ.empty:
        return summ
    method_rank = {method: i for i, method in enumerate(order)}
    summ["__rank"] = summ["method"].map(method_rank).fillna(1e9)
    return summ.sort_values(["__rank", cfg.core_x_col]).drop(columns="__rank").reset_index(drop=True)


def plot_core_metric_figure(
    summ: pd.DataFrame,
    cfg: SimulationSetConfig,
    *,
    title: Optional[str] = None,
    output_suffix: Optional[str] = None,
    method_order: Optional[Sequence[str]] = None,
    method_name_map: Optional[dict[str, str]] = None,
    save: bool = True,
) -> plt.Figure:
    if summ.empty:
        raise ValueError(f"{cfg.title}: no summarized data to plot.")

    label_map = {**METHOD_NAME_MAP, **(method_name_map or {})}
    order = list(method_order or METHOD_ORDER)
    x_values = sorted(pd.unique(summ[cfg.core_x_col].dropna()))
    methods_present = [method for method in order if (summ["method"] == method).any()]
    if not methods_present:
        methods_present = sorted(pd.unique(summ["method"]))

    fig, axes = plt.subplots(
        1,
        3,
        figsize=CORE_PLOT_PARAMS["figsize"],
        dpi=CORE_PLOT_PARAMS["dpi"],
        sharex=True,
    )
    fig.suptitle(title or cfg.title, fontsize=CORE_PLOT_PARAMS["suptitle_fontsize"], y=CORE_PLOT_PARAMS["suptitle_y"])

    for ax, (metric, metric_label) in zip(axes, METRICS):
        if metric == "fdr":
            ax.axhline(ALPHA, color="red", linestyle="--", linewidth=1.5, zorder=0)

        for method in methods_present:
            sub = summ[summ["method"] == method].sort_values(cfg.core_x_col)
            sub = sub.set_index(cfg.core_x_col).reindex(x_values).reset_index()

            x = sub[cfg.core_x_col].to_numpy(dtype=float)
            y = sub[f"{metric}_mean"].to_numpy(dtype=float)
            lo = sub[f"{metric}_lo"].to_numpy(dtype=float)
            hi = sub[f"{metric}_hi"].to_numpy(dtype=float)
            n = sub[f"{metric}_n"].to_numpy(dtype=float)

            lo_adj = np.where(n >= 3, lo, y)
            hi_adj = np.where(n >= 3, hi, y)
            yerr = np.vstack([y - lo_adj, hi_adj - y])

            ax.errorbar(
                x,
                y,
                yerr=yerr,
                fmt="o-",
                linewidth=2.0,
                markersize=5.5,
                capsize=CORE_PLOT_PARAMS["capsize"],
                elinewidth=CORE_PLOT_PARAMS["err_linewidth"],
                label=label_map.get(method, method),
            )

        ax.set_ylabel(metric_label, fontsize=CORE_PLOT_PARAMS["axis_label_fontsize"], labelpad=10)
        ax.tick_params(axis="both", labelsize=CORE_PLOT_PARAMS["tick_label_fontsize"])
        ax.grid(False)

    fig.supxlabel(cfg.core_x_label, fontsize=CORE_PLOT_PARAMS["axis_label_fontsize"], y=0.1)

    handles, labels_ = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels_,
        fontsize=CORE_PLOT_PARAMS["legend_fontsize"],
        frameon=True,
        loc="upper center",
        ncol=1 if len(labels_) > 2 else len(labels_),
        bbox_to_anchor=(0.5, CORE_PLOT_PARAMS["legend_y"]),
    )
    fig.tight_layout(rect=CORE_PLOT_PARAMS["tight_rect"])

    if save:
        suffix = output_suffix or cfg.output_suffix
        out_png = BASE_DIR / f"standard_metrics_{suffix}.png"
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        print(f"Saved -> {out_png}")
    return fig


def build_core_method_comparison(
    comparison: MethodComparisonConfig,
    results_by_key: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    parts: list[pd.DataFrame] = []
    for member in comparison.members:
        df = results_by_key.get(member.cfg_key, pd.DataFrame())
        if df.empty or "method" not in df.columns:
            continue
        sub = df[df["method"] == member.source_method].copy()
        if sub.empty:
            continue
        sub["method"] = member.plot_method
        sub["method_label"] = member.label
        sub["comparison"] = comparison.key
        parts.append(sub)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


In [9]:
core_summaries: dict[str, pd.DataFrame] = {}
for cfg in SIMULATION_SETS:
    print(f"Plotting standard metrics: {cfg.title}")
    core_summaries[cfg.key] = summarize_core_metrics(core_results[cfg.key], cfg)
    display(core_summaries[cfg.key].head())
    if core_summaries[cfg.key].empty:
        print(f"Skipping {cfg.title}: no standard metrics to plot.")
        continue
    plot_core_metric_figure(core_summaries[cfg.key], cfg)
    plt.show()

core_comparison_results: dict[str, pd.DataFrame] = {}
core_comparison_summaries: dict[str, pd.DataFrame] = {}
for comparison in METHOD_COMPARISONS:
    base_cfg = SIM_BY_KEY[comparison.base_cfg_key]
    print(f"Plotting standard metric comparison: {comparison.title}")
    comp_df = build_core_method_comparison(comparison, core_results)
    core_comparison_results[comparison.key] = comp_df
    if comp_df.empty:
        print(f"Skipping {comparison.title}: no comparison rows were available.")
        continue
    core_comparison_summaries[comparison.key] = summarize_core_metrics(
        comp_df,
        base_cfg,
        method_order=comparison.method_order,
        method_name_map=comparison.method_name_map,
    )
    display(core_comparison_summaries[comparison.key].head())
    plot_core_metric_figure(
        core_comparison_summaries[comparison.key],
        base_cfg,
        title=comparison.title,
        output_suffix=comparison.output_suffix,
        method_order=comparison.method_order,
        method_name_map=comparison.method_name_map,
    )
    plt.show()

# Backward-compatible alias used by the original notebook.
summ = core_summaries["original"]


Plotting standard metrics: Original simulations


,method,method_label,src_n,n_points,fdr_mean,fdr_lo,fdr_hi,fdr_n,power_mean,power_lo,power_hi,power_n,auprc_mean,auprc_lo,auprc_hi,auprc_n
0,scdrs+_conditional,scDRS-FM,1,15,0.083025,0.056680,0.109371,15,0.882552,0.842231,0.922872,15,0.933575,0.919544,0.947605,15
1,scdrs+_conditional,scDRS-FM,2,15,0.045670,0.024560,0.066780,15,0.748417,0.663963,0.832870,15,0.924455,0.890032,0.958878,15
2,scdrs+_conditional,scDRS-FM,3,15,0.035940,0.026174,0.045707,15,0.593177,0.531954,0.654401,15,0.929176,0.916675,0.941676,15
3,scdrs+_conditional,scDRS-FM,4,15,0.026330,0.013331,0.039329,15,0.363246,0.313544,0.412947,15,0.917113,0.902192,0.932033,15
4,scdrs+_conditional,scDRS-FM,5,15,0.021711,0.011230,0.032192,15,0.214763,0.157072,0.272455,15,0.904240,0.887233,0.921248,15


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/standard_metrics_original.png
Plotting standard metrics: Cell-percent simulations


,method,method_label,min_cell_percent,n_points,fdr_mean,fdr_lo,fdr_hi,fdr_n,power_mean,power_lo,power_hi,power_n,auprc_mean,auprc_lo,auprc_hi,auprc_n
0,scdrs+_conditional,scDRS-FM,1,93,0.134748,0.109449,0.160047,93,0.759616,0.720218,0.799013,93,0.875856,0.855702,0.896010,93
1,scdrs+_conditional,scDRS-FM,2,57,0.107898,0.076747,0.139049,57,0.688293,0.638971,0.737614,57,0.863303,0.834826,0.891780,57
2,scdrs+_conditional,scDRS-FM,3,30,0.048685,0.034828,0.062543,30,0.697719,0.646544,0.748894,30,0.917150,0.897813,0.936488,30
3,scdrs+_conditional,scDRS-FM,4,18,0.038324,0.026860,0.049789,18,0.607662,0.533554,0.681770,18,0.918873,0.901438,0.936309,18
4,scdrs+_conditional,scDRS-FM,5,15,0.032106,0.019433,0.044779,15,0.553611,0.481855,0.625367,15,0.919709,0.906024,0.933394,15


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/standard_metrics_sims_percent.png
Plotting standard metrics: Causal-gene simulations


,method,method_label,overlap_k,n_points,fdr_mean,fdr_lo,fdr_hi,fdr_n,power_mean,power_lo,power_hi,power_n,auprc_mean,auprc_lo,auprc_hi,auprc_n
0,scdrs+_conditional,scDRS-FM,25,15,0.055538,0.030703,0.080373,15,0.293989,0.215881,0.372097,15,0.865814,0.822928,0.908701,15
1,scdrs+_conditional,scDRS-FM,50,15,0.028100,0.016071,0.040129,15,0.543465,0.469830,0.617100,15,0.917732,0.896243,0.939221,15
2,scdrs+_conditional,scDRS-FM,75,15,0.032399,0.019952,0.044847,15,0.640156,0.578338,0.701974,15,0.927576,0.911811,0.943340,15
3,scdrs+_conditional,scDRS-FM,100,15,0.021581,0.015097,0.028064,15,0.633812,0.595174,0.672449,15,0.929436,0.919084,0.939788,15
4,scdrs+_marginal,scDRS-FM marginal,25,15,0.197515,0.140481,0.254549,15,0.695956,0.593654,0.798258,15,0.861737,0.812132,0.911343,15


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/standard_metrics_sims_genes.png
Plotting standard metric comparison: Denoising comparison: MAGIC vs KNN


,method,method_label,src_n,n_points,fdr_mean,fdr_lo,fdr_hi,fdr_n,power_mean,power_lo,power_hi,power_n,auprc_mean,auprc_lo,auprc_hi,auprc_n
0,magic,MAGIC,1,15,0.083025,0.056680,0.109371,15,0.882552,0.842231,0.922872,15,0.933575,0.919544,0.947605,15
1,magic,MAGIC,2,15,0.045670,0.024560,0.066780,15,0.748417,0.663963,0.832870,15,0.924455,0.890032,0.958878,15
2,magic,MAGIC,3,15,0.035940,0.026174,0.045707,15,0.593177,0.531954,0.654401,15,0.929176,0.916675,0.941676,15
3,magic,MAGIC,4,15,0.026330,0.013331,0.039329,15,0.363246,0.313544,0.412947,15,0.917113,0.902192,0.932033,15
4,magic,MAGIC,5,15,0.021711,0.011230,0.032192,15,0.214763,0.157072,0.272455,15,0.904240,0.887233,0.921248,15


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/standard_metrics_magic_vs_knn.png


## Independent-signal evaluation

This section evaluates independent signals for the configured scDRS-FM result sets. The default signal assignment is `independent_signal_multi` when that column exists, and otherwise `independent_signal`.


In [10]:
# Columns in predictions files that may contain the causal cluster labels.
CAUSAL_LABEL_COL_CANDIDATES = (
    "trait_donor_clusters",
    "donors",
    "donor_clusters",
    "causal_clusters",
    "causal_leiden",
    "causal_leiden_labels",
    "true_clusters",
    "trait_donor_clusters_other",
)
GENESET_ID_COL_CANDIDATES = (
    "geneset_file",
    "gs_file",
    "geneset",
    "geneset_filename",
    "geneset_path",
)


def _find_first_existing_recursive(out_root: Path, candidates: Sequence[str]) -> Optional[Path]:
    for name in candidates:
        direct = out_root / name
        if direct.exists():
            return direct
        try:
            hit = next(out_root.rglob(name))
            if hit.exists():
                return hit
        except StopIteration:
            pass
    return None


def _find_marginal_file_recursive(out_root: Path, prefixes: Sequence[str]) -> Optional[Path]:
    candidates: list[str] = []
    for pref in prefixes:
        candidates.extend([f"{pref}.marginal_score.gz", f"{pref}.marginal.score.gz", f"{pref}.score.gz"])
    return _find_first_existing_recursive(out_root, candidates)


def _find_conditional_file_recursive(out_root: Path, prefixes: Sequence[str]) -> Optional[Path]:
    candidates: list[str] = []
    for pref in prefixes:
        candidates.extend([
            f"{pref}.conditional.tagging_score.gz",
            f"{pref}.tagging_score.gz",
            f"{pref}.conditional_score.gz",
        ])
    return _find_first_existing_recursive(out_root, candidates)


def _get_causal_labels_from_predictions(
    pred_df: pd.DataFrame,
    cfg: SimulationSetConfig,
    *,
    gs_path: Path,
    meta: dict[str, Any],
    trait: str,
) -> Optional[set[str]]:
    """Find the matching prediction-table row and return its causal label set."""
    df = pred_df.copy()
    for key in cfg.prediction_merge_keys:
        if key in df.columns and key in meta:
            vals = pd.to_numeric(df[key], errors="coerce")
            mask = vals == float(meta[key])
            if mask.any():
                df = df.loc[mask].copy()

    matched = None
    for id_col in GENESET_ID_COL_CANDIDATES:
        if id_col in df.columns:
            col = df[id_col].astype(str)
            name = gs_path.name
            stem = gs_path.stem
            mask = (col == name) | (col == stem) | col.str.endswith("/" + name) | col.str.endswith("\\" + name)
            if mask.any():
                matched = df.loc[mask]
                break

    if matched is None:
        for trait_col in ("trait", "trait_name", "trait_in_file"):
            if trait_col in df.columns:
                mask = df[trait_col].astype(str) == str(trait)
                if mask.any():
                    matched = df.loc[mask]
                    break

    if matched is None or matched.empty:
        return None

    label_col = next((col for col in CAUSAL_LABEL_COL_CANDIDATES if col in matched.columns), None)
    if label_col is None:
        return None
    return _parse_label_set(matched.iloc[0][label_col])


def _metacell_marg_sig_fraction(cell_ids: str, sig_cells_set: set[str]) -> float:
    if not cell_ids or str(cell_ids).lower() == "nan":
        return 0.0
    cells = [c for c in str(cell_ids).split(",") if c]
    if not cells:
        return 0.0
    return sum(1 for c in cells if c in sig_cells_set) / float(len(cells))


def select_metacells_by_pvalue_and_marginal(
    df_cond: pd.DataFrame,
    *,
    pcol: str,
    sig_cells_marg: set[str],
    fdr_alpha: float,
    min_frac_cells_marg_sig: float,
) -> tuple[pd.DataFrame, int]:
    """Select significant metacells before applying any signal-assignment filter."""
    if pcol not in df_cond.columns or "cell_ids" not in df_cond.columns:
        return df_cond.iloc[0:0].copy(), 0

    df = _normalize_metacell_index_to_int(df_cond)
    p = pd.to_numeric(df[pcol], errors="coerce").to_numpy(dtype=np.float64)
    rej = bh_fdr_mask(p, alpha=fdr_alpha)
    if rej.sum() == 0:
        return df.iloc[0:0].copy(), 0

    df_sel = df.loc[df.index[rej]].copy()
    frac_vec = np.array(
        [_metacell_marg_sig_fraction(s, sig_cells_marg) for s in df_sel["cell_ids"].astype(str).to_numpy()],
        dtype=np.float64,
    )
    df_sel["metacell_frac_cells_marg_sig"] = frac_vec
    df_sel = df_sel[df_sel["metacell_frac_cells_marg_sig"] >= float(min_frac_cells_marg_sig)].copy()
    return df_sel, int(df_sel.shape[0])


def pick_primary_signal_col(df_cond: pd.DataFrame) -> Optional[str]:
    for col in PRIMARY_SIGNAL_COL_CANDIDATES:
        if col in df_cond.columns:
            return col
    return None


def restrict_to_assigned_signal(df_sel: pd.DataFrame, signal_col: str) -> pd.DataFrame:
    if df_sel is None or df_sel.empty or signal_col not in df_sel.columns:
        return df_sel.iloc[0:0].copy() if isinstance(df_sel, pd.DataFrame) else pd.DataFrame()
    sig_as_int = pd.to_numeric(df_sel[signal_col], errors="coerce")
    keep = sig_as_int.notna() & (sig_as_int >= 0)
    out = df_sel.loc[keep].copy()
    out[signal_col] = sig_as_int.loc[keep].astype(int)
    return out


def _cells_in_selected_metacells(df_sel: pd.DataFrame) -> set[str]:
    if df_sel is None or df_sel.empty or "cell_ids" not in df_sel.columns:
        return set()
    out: set[str] = set()
    for cell_ids in df_sel["cell_ids"].astype(str).tolist():
        if cell_ids and cell_ids.lower() != "nan":
            out.update([c for c in cell_ids.split(",") if c])
    return out


def _majority_label_for_metacell(cell_ids: str, cell_to_label: dict[str, str]) -> Optional[str]:
    if not cell_ids or str(cell_ids).lower() == "nan":
        return None
    labels_here = [cell_to_label.get(c) for c in str(cell_ids).split(",") if c]
    labels_here = [str(label) for label in labels_here if label is not None and str(label).lower() != "nan"]
    if not labels_here:
        return None
    counts = pd.Series(labels_here).value_counts()
    return str(counts.sort_index().sort_values(ascending=False).index[0])


def _signal_majority_label_from_metacells(labels_for_signal: Sequence[str]) -> Optional[str]:
    labels_for_signal = [str(x) for x in labels_for_signal if x is not None and str(x).lower() != "nan"]
    if not labels_for_signal:
        return None
    counts = pd.Series(labels_for_signal).value_counts()
    return str(counts.sort_index().sort_values(ascending=False).index[0])


def _count_discovered_labels_celllevel_restricted(
    *,
    intersection_cells: set[str],
    cell_to_label: dict[str, str],
    total_counts: dict[str, int],
    min_frac: float,
    allowed_labels: set[str],
) -> int:
    allowed = {str(x) for x in allowed_labels}
    if not intersection_cells or not allowed:
        return 0

    hit: dict[str, int] = {}
    for cell in intersection_cells:
        label = cell_to_label.get(cell)
        if label is not None and str(label) in allowed:
            hit[str(label)] = hit.get(str(label), 0) + 1

    discovered = 0
    for label in allowed:
        denom = int(total_counts.get(str(label), 0))
        if denom > 0 and (hit.get(str(label), 0) / float(denom)) >= float(min_frac):
            discovered += 1
    return discovered


def compute_independent_signal_metrics(
    df_sel: pd.DataFrame,
    *,
    signal_col: str,
    cell_to_leiden: dict[str, str],
    total_leiden_counts: dict[str, int],
    sig_cells_marg: set[str],
    causal_leiden_labels: set[str],
    discovery_min_frac_celllevel: float,
) -> dict[str, Any]:
    out: dict[str, Any] = {
        "n_indep_signals_selected": 0.0,
        "metacell_self_agreement_majority_leiden": np.nan,
        "n_false_indep_signals_selected": 0.0,
        "n_leiden_clusters_discovered_in_truth": 0.0,
    }
    if df_sel is None or df_sel.empty or signal_col not in df_sel.columns:
        return out

    signals = np.unique(pd.to_numeric(df_sel[signal_col], errors="coerce").dropna().astype(int).to_numpy())
    out["n_indep_signals_selected"] = float(signals.size)

    causal_set = {str(x) for x in causal_leiden_labels}
    cond_cells = _cells_in_selected_metacells(df_sel)
    inter_cells = cond_cells.intersection(sig_cells_marg)
    out["n_leiden_clusters_discovered_in_truth"] = float(
        _count_discovered_labels_celllevel_restricted(
            intersection_cells=inter_cells,
            cell_to_label=cell_to_leiden,
            total_counts=total_leiden_counts,
            min_frac=discovery_min_frac_celllevel,
            allowed_labels=causal_set,
        )
    )

    mc_major_leiden = {
        int(mid): _majority_label_for_metacell(cell_ids, cell_to_leiden)
        for mid, cell_ids in zip(df_sel.index.to_numpy(dtype=int), df_sel["cell_ids"].astype(str).to_numpy())
    }

    total_used = 0
    total_match = 0
    n_false = 0
    for signal in signals.tolist():
        sub = df_sel[pd.to_numeric(df_sel[signal_col], errors="coerce").astype("Int64") == int(signal)]
        labs = [mc_major_leiden.get(int(mid)) for mid in sub.index.to_numpy(dtype=int)]
        labs = [str(label) for label in labs if label is not None and str(label).lower() != "nan"]
        if not labs:
            continue
        majority = _signal_majority_label_from_metacells(labs)
        total_used += len(labs)
        total_match += sum(1 for label in labs if str(label) == str(majority))
        if not any(str(label) in causal_set for label in labs):
            n_false += 1

    out["n_false_indep_signals_selected"] = float(n_false)
    out["metacell_self_agreement_majority_leiden"] = float(total_match) / float(total_used) if total_used else np.nan
    return out


In [11]:
def evaluate_independent_signal_set(
    cfg: SimulationSetConfig,
    *,
    obs_names: pd.Index,
    cell_to_leiden: dict[str, str],
    total_leiden_counts: dict[str, int],
) -> pd.DataFrame:
    pred_df = load_predictions(cfg)
    out_root = choose_existing_root(cfg.indep_root_candidates, cfg.scdrsplus_root)

    rows: list[dict[str, Any]] = []
    skipped = 0
    missing_truth = 0
    missing_cond = 0
    missing_marg = 0
    n_files_with_neg1 = 0

    for gs_path in geneset_files(cfg):
        try:
            meta = parse_geneset_filename(gs_path, cfg)
            trait = _read_trait_from_gs(gs_path)
            trait_info = _parse_trait_info(trait)
        except Exception:
            skipped += 1
            continue

        causal_labels = _get_causal_labels_from_predictions(
            pred_df,
            cfg,
            gs_path=gs_path,
            meta=meta,
            trait=trait,
        )
        if causal_labels is None or len(causal_labels) == 0:
            causal_labels = set(trait_info["donors"])
        if not causal_labels:
            missing_truth += 1
            continue

        prefixes = [trait, gs_path.name, gs_path.stem]
        cond_path = _find_conditional_file_recursive(out_root, prefixes)
        marg_path = _find_marginal_file_recursive(out_root, prefixes)
        if cond_path is None:
            missing_cond += 1
            continue
        if marg_path is None:
            missing_marg += 1
            continue

        try:
            sig_cells_marg, _ = _read_marginal(marg_path, obs_names, alpha=FDR_ALPHA)
            sig_cells_marg_set = set(sig_cells_marg.astype(str).tolist())
            df_cond = pd.read_csv(cond_path, sep="\t", compression="gzip", index_col=0)
        except Exception:
            skipped += 1
            continue

        primary_signal_col = pick_primary_signal_col(df_cond)
        has_neg1 = False
        if primary_signal_col is not None:
            signal_numeric = pd.to_numeric(df_cond[primary_signal_col], errors="coerce")
            has_neg1 = bool((signal_numeric == -1).any())
            if has_neg1:
                n_files_with_neg1 += 1

        base_row: dict[str, Any] = {
            "simulation_set": cfg.key,
            "cluster": int(meta["cluster"]),
            "replicate": int(meta["replicate"]),
            "src_n": int(meta["src_n"]),
            "causal_clusters": int(meta["src_n"]),
            "overlap_k": int(meta["overlap_k"]),
            "causal_genes_per_cluster": int(meta["overlap_k"]),
            "geneset_file": gs_path.name,
            "geneset_path": str(gs_path),
            "method": "scdrs+_conditional",
            "conditional_file": str(cond_path),
            "marginal_file": str(marg_path),
            "causal_leiden_labels": ";".join(sorted({str(x) for x in causal_labels})),
            "primary_signal_col": primary_signal_col,
            "has_neg1_indep_signal": bool(has_neg1),
            "indep_root": str(out_root),
        }
        if "min_cell_percent" in meta:
            base_row["min_cell_percent"] = int(meta["min_cell_percent"])

        if primary_signal_col is None or "cell_ids" not in df_cond.columns:
            rows.append({
                **base_row,
                "n_indep_signals_selected": np.nan,
                "metacell_self_agreement_majority_leiden": np.nan,
                "n_false_indep_signals_selected": np.nan,
                "n_true_indep_signals_selected": np.nan,
                "n_metacells_selected_total": 0,
                "n_leiden_clusters_discovered_in_truth": np.nan,
            })
            continue

        try:
            pcol = _pick_first_existing_col(df_cond, COND_PVAL_COL_CANDIDATES, what="conditional p-value")
        except Exception:
            rows.append({
                **base_row,
                "n_indep_signals_selected": np.nan,
                "metacell_self_agreement_majority_leiden": np.nan,
                "n_false_indep_signals_selected": np.nan,
                "n_true_indep_signals_selected": np.nan,
                "n_metacells_selected_total": 0,
                "n_leiden_clusters_discovered_in_truth": np.nan,
            })
            continue

        df_base, n_metacells_before_signal_filter = select_metacells_by_pvalue_and_marginal(
            df_cond,
            pcol=pcol,
            sig_cells_marg=sig_cells_marg_set,
            fdr_alpha=FDR_ALPHA,
            min_frac_cells_marg_sig=MIN_FRAC_CELLS_MARG_SIG,
        )
        df_sel = restrict_to_assigned_signal(df_base, primary_signal_col)

        metrics = compute_independent_signal_metrics(
            df_sel,
            signal_col=primary_signal_col,
            cell_to_leiden=cell_to_leiden,
            total_leiden_counts=total_leiden_counts,
            sig_cells_marg=sig_cells_marg_set,
            causal_leiden_labels=causal_labels,
            discovery_min_frac_celllevel=DISCOVERY_MIN_FRAC_CELLLEVEL,
        )

        n_true = metrics["n_indep_signals_selected"] - metrics["n_false_indep_signals_selected"]
        rows.append({
            **base_row,
            **metrics,
            "n_true_indep_signals_selected": float(n_true),
            "n_metacells_selected_total": int(n_metacells_before_signal_filter),
        })

    out = pd.DataFrame(rows)
    if out.empty:
        print(f"{cfg.title}: no independent-signal rows were evaluated.")
        return out

    out["n_indep_signals_selected_int"] = (
        pd.to_numeric(out["n_indep_signals_selected"], errors="coerce").round().astype("Int64").astype(float)
    )
    out["n_leiden_clusters_discovered_in_truth_int"] = (
        pd.to_numeric(out["n_leiden_clusters_discovered_in_truth"], errors="coerce").round().astype("Int64").astype(float)
    )

    sort_cols = [c for c in [cfg.indep_x_col, "cluster", "replicate", "method"] if c in out.columns]
    out = out.sort_values(sort_cols).reset_index(drop=True)

    if skipped:
        print(f"{cfg.title}: skipped {skipped} files during independent-signal evaluation.")
    if missing_truth:
        print(f"{cfg.title}: missing truth labels for {missing_truth} genesets.")
    if missing_cond:
        print(f"{cfg.title}: missing conditional outputs for {missing_cond} genesets.")
    if missing_marg:
        print(f"{cfg.title}: missing marginal outputs for {missing_marg} genesets.")
    if n_files_with_neg1:
        print(f"{cfg.title}: {n_files_with_neg1} conditional files contained a -1 primary signal assignment.")

    return out


In [12]:
# Reuse the labels loaded earlier as the simulation truth labels.
cell_to_leiden = dict(zip(obs_names.tolist(), labels.tolist()))
total_leiden_counts = pd.Series(labels, index=obs_names).value_counts().astype(int).to_dict()

# Compute a separate Leiden clustering of the dataset for the scDRS w/ Leiden baseline.
scdrs_leiden_series = compute_or_load_scdrs_leiden_labels(
    H5AD_PATH,
    obs_names=obs_names,
    resolution=SCDRS_LEIDEN_RESOLUTION,
    key_added=SCDRS_LEIDEN_KEY,
)
cell_to_scdrs_leiden = scdrs_leiden_series.dropna().astype(str).to_dict()
total_scdrs_leiden_counts = scdrs_leiden_series.dropna().astype(str).value_counts().astype(int).to_dict()

independent_results: dict[str, pd.DataFrame] = {}
for cfg in ALL_EVALUATION_SETS:
    print(f"Evaluating independent signals: {cfg.title}")
    independent_results[cfg.key] = evaluate_independent_signal_set(
        cfg,
        obs_names=obs_names,
        cell_to_leiden=cell_to_leiden,
        total_leiden_counts=total_leiden_counts,
    )

# Backward-compatible alias used by the original notebook.
results_for_plot = independent_results["original"].copy()
if EXCLUDE_FILES_WITH_NEG1_INDEP_SIGNAL_FROM_PLOTS and "has_neg1_indep_signal" in results_for_plot.columns:
    results_for_plot = results_for_plot.loc[~results_for_plot["has_neg1_indep_signal"]].copy()
results_for_plot.head()


/tmp/ipykernel_2488/329960033.py:260: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=float(resolution), key_added=key_added)


Evaluating independent signals: Original simulations


Evaluating independent signals: Cell-percent simulations


Evaluating independent signals: Causal-gene simulations


Evaluating independent signals: Original simulations — KNN denoising


,simulation_set,cluster,replicate,src_n,causal_clusters,overlap_k,causal_genes_per_cluster,geneset_file,geneset_path,method,...,has_neg1_indep_signal,indep_root,n_indep_signals_selected,metacell_self_agreement_majority_leiden,n_false_indep_signals_selected,n_leiden_clusters_discovered_in_truth,n_true_indep_signals_selected,n_metacells_selected_total,n_indep_signals_selected_int,n_leiden_clusters_discovered_in_truth_int
0,original,0,0,1,1,50,50,TMS_FACS_0_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,1.0,0.936842,0.0,1.0,1.0,95,1.0,1.0
1,original,0,1,1,1,50,50,TMS_FACS_0_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,2.0,0.953488,0.0,1.0,2.0,86,2.0,1.0
2,original,0,2,1,1,50,50,TMS_FACS_0_2_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,3.0,0.940594,1.0,1.0,2.0,101,3.0,1.0
3,original,1,0,1,1,50,50,TMS_FACS_1_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,1.0,0.988636,0.0,1.0,1.0,88,1.0,1.0
4,original,1,1,1,1,50,50,TMS_FACS_1_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,1.0,0.958333,0.0,1.0,1.0,72,1.0,1.0


## Independent-signal 3-panel figures

The same three-panel independent-signal summary is produced separately for each simulation family. The MAGIC-vs-KNN figure is generated through the generic method-comparison path.


In [13]:
INDEP_FIG_W = 18
INDEP_FIG_H = 5.2
FONT_FAMILY = "DejaVu Sans"
FONT_SIZE_BASE = 14
FONT_SIZE_TITLE = 18
FONT_SIZE_LABEL = 16
FONT_SIZE_TICK = 14
FONT_SIZE_LEGEND = 14
LINEWIDTH = 2.2
MARKERSIZE = 7.5
CAPSIZE = 4
BAR_WIDTH = 0.65
COLOR_TRUE = "tab:blue"
COLOR_FALSE = "tab:red"

plt.rcParams.update({
    "font.family": FONT_FAMILY,
    "font.size": FONT_SIZE_BASE,
    "axes.titlesize": FONT_SIZE_TITLE,
    "axes.labelsize": FONT_SIZE_LABEL,
    "xtick.labelsize": FONT_SIZE_TICK,
    "ytick.labelsize": FONT_SIZE_TICK,
    "legend.fontsize": FONT_SIZE_LEGEND,
})


def _mean_ci95_t(vals: np.ndarray) -> tuple[float, float, int]:
    x = np.asarray(vals, dtype=np.float64)
    x = x[np.isfinite(x)]
    n = int(x.size)
    if n == 0:
        return np.nan, np.nan, 0
    m = float(np.mean(x))
    if n < 2:
        return m, np.nan, n
    se = float(np.std(x, ddof=1) / np.sqrt(n))
    tcrit = float(stats.t.ppf(0.975, df=n - 1))
    return m, tcrit * se, n


def _aggregate_mean_ci(df: pd.DataFrame, *, x_col: str, y_col: str) -> pd.DataFrame:
    d = df.dropna(subset=[x_col, y_col]).copy()
    if d.empty:
        return pd.DataFrame(columns=["method", x_col, "mean", "ci_half", "n"])
    rows = []
    for (method, xval), group in d.groupby(["method", x_col], sort=True):
        mean, half, n = _mean_ci95_t(group[y_col].to_numpy(dtype=np.float64))
        rows.append({"method": method, x_col: xval, "mean": mean, "ci_half": half, "n": n})
    return pd.DataFrame(rows).sort_values(["method", x_col]).reset_index(drop=True)


def _aggregate_true_false_by_x(df: pd.DataFrame, *, x_col: str, true_col: str, false_col: str) -> pd.DataFrame:
    d = df.dropna(subset=[x_col, true_col, false_col]).copy()
    if d.empty:
        return pd.DataFrame(columns=[x_col, "true_mean", "true_ci", "false_mean", "false_ci", "n"])
    rows = []
    for xval, group in d.groupby(x_col, sort=True):
        t_mean, t_ci, n_t = _mean_ci95_t(group[true_col].to_numpy(dtype=np.float64))
        f_mean, f_ci, n_f = _mean_ci95_t(group[false_col].to_numpy(dtype=np.float64))
        rows.append({x_col: xval, "true_mean": t_mean, "true_ci": t_ci, "false_mean": f_mean, "false_ci": f_ci, "n": min(n_t, n_f)})
    return pd.DataFrame(rows).sort_values(x_col).reset_index(drop=True)


def _plot_stacked_true_false_bar(
    ax: plt.Axes,
    results_df: pd.DataFrame,
    *,
    x_col: str,
    true_col: str,
    false_col: str,
    title: str,
    xlabel: str,
    ylabel: str,
) -> None:
    method = "scdrs+_conditional"
    df = results_df[results_df["method"] == method].copy()
    agg = _aggregate_true_false_by_x(df, x_col=x_col, true_col=true_col, false_col=false_col)
    if agg.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title, fontsize=FONT_SIZE_TITLE)
        return

    xvals = agg[x_col].to_list()
    pos = np.arange(len(xvals), dtype=float)
    true_mean = agg["true_mean"].to_numpy(dtype=float)
    false_mean = agg["false_mean"].to_numpy(dtype=float)

    ax.bar(pos, true_mean, width=BAR_WIDTH, color=COLOR_TRUE, label="True signals")
    ax.bar(pos, false_mean, bottom=true_mean, width=BAR_WIDTH, color=COLOR_FALSE, label="False signals")
    ax.errorbar(pos, true_mean, yerr=agg["true_ci"].to_numpy(dtype=float), fmt="none", capsize=CAPSIZE, linewidth=LINEWIDTH, color="k")
    ax.errorbar(pos, true_mean + false_mean, yerr=agg["false_ci"].to_numpy(dtype=float), fmt="none", capsize=CAPSIZE, linewidth=LINEWIDTH, color="k")

    ax.set_xticks(pos)
    ax.set_xticklabels([str(int(v)) if float(v).is_integer() else str(v) for v in xvals])
    ax.set_xlabel(xlabel, fontsize=FONT_SIZE_LABEL)
    ax.set_ylabel(ylabel, fontsize=FONT_SIZE_LABEL)
    ax.set_title(title, fontsize=FONT_SIZE_TITLE)
    ax.tick_params(axis="both", labelsize=FONT_SIZE_TICK)
    ax.legend(frameon=False, fontsize=FONT_SIZE_LEGEND)


def _plot_mean_ci_lines_by_method(
    ax: plt.Axes,
    results_df: pd.DataFrame,
    *,
    x_col: str,
    y_col: str,
    title: str,
    xlabel: str,
    ylabel: str,
    method_order: Sequence[str] = ("scdrs+_conditional",),
    method_name_map: Optional[dict[str, str]] = None,
    ylim: Optional[tuple[float, float]] = None,
    xticklabels: Optional[list[str]] = None,
    legend: bool = False,
) -> None:
    label_map = {**METHOD_NAME_MAP, **(method_name_map or {})}
    agg = _aggregate_mean_ci(results_df, x_col=x_col, y_col=y_col)
    if agg.empty:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(title, fontsize=FONT_SIZE_TITLE)
        return

    xvals = sorted({float(v) for v in agg[x_col].dropna().unique()})
    ax.set_xticks(xvals)
    if xticklabels is not None and len(xticklabels) == len(xvals):
        ax.set_xticklabels(xticklabels)

    for method in method_order:
        sub = agg[agg["method"] == method].copy()
        if sub.empty:
            continue
        ax.errorbar(
            sub[x_col].to_numpy(dtype=float),
            sub["mean"].to_numpy(dtype=float),
            yerr=sub["ci_half"].to_numpy(dtype=float),
            fmt="o-",
            capsize=CAPSIZE,
            linewidth=LINEWIDTH,
            markersize=MARKERSIZE,
            label=label_map.get(method, method),
        )

    ax.set_xlabel(xlabel, fontsize=FONT_SIZE_LABEL)
    ax.set_ylabel(ylabel, fontsize=FONT_SIZE_LABEL)
    ax.set_title(title, fontsize=FONT_SIZE_TITLE)
    ax.tick_params(axis="both", labelsize=FONT_SIZE_TICK)
    if ylim is not None:
        ax.set_ylim(*ylim)
    if legend:
        ax.legend(frameon=False, fontsize=FONT_SIZE_LEGEND)


def independent_results_for_plot(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if EXCLUDE_FILES_WITH_NEG1_INDEP_SIGNAL_FROM_PLOTS and "has_neg1_indep_signal" in out.columns:
        out = out.loc[~out["has_neg1_indep_signal"]].copy()
    return out


def plot_independent_signal_figure(df: pd.DataFrame, cfg: SimulationSetConfig, *, save: bool = True) -> plt.Figure:
    results_df = independent_results_for_plot(df)
    if results_df.empty:
        raise ValueError(f"{cfg.title}: no independent-signal data to plot.")

    x_col = cfg.indep_x_col
    fig, axes = plt.subplots(1, 3, figsize=(INDEP_FIG_W, INDEP_FIG_H))
    fig.suptitle(cfg.title, fontsize=FONT_SIZE_TITLE + 2, y=1.02)

    _plot_stacked_true_false_bar(
        axes[0],
        results_df,
        x_col=x_col,
        true_col="n_true_indep_signals_selected",
        false_col="n_false_indep_signals_selected",
        title="Independent signals detected",
        xlabel=cfg.indep_x_label,
        ylabel="Indep. signals detected",
    )
    _plot_mean_ci_lines_by_method(
        axes[1],
        results_df,
        x_col=x_col,
        y_col="metacell_self_agreement_majority_leiden",
        title="Signal self agreement",
        xlabel=cfg.indep_x_label,
        ylabel="Within signal purity",
        ylim=(0.0, 1.0),
    )

    method_df = results_df[results_df["method"] == "scdrs+_conditional"].copy()
    tick_counts = method_df["n_leiden_clusters_discovered_in_truth_int"].value_counts().to_dict()
    agg3 = _aggregate_mean_ci(
        results_df,
        x_col="n_leiden_clusters_discovered_in_truth_int",
        y_col="n_indep_signals_selected_int",
    )
    xvals3 = sorted({float(v) for v in agg3["n_leiden_clusters_discovered_in_truth_int"].dropna().unique()}) if not agg3.empty else []
    xticklabels3 = [f"{int(x)}\n(n={int(tick_counts.get(float(x), tick_counts.get(int(x), 0)))})" for x in xvals3]

    _plot_mean_ci_lines_by_method(
        axes[2],
        results_df,
        x_col="n_leiden_clusters_discovered_in_truth_int",
        y_col="n_indep_signals_selected_int",
        title="Indep. signals vs. causal clusters discovered",
        xlabel="Causal clusters discovered",
        ylabel="Indep. signals",
        xticklabels=xticklabels3 if xticklabels3 else None,
    )
    x0, x1 = axes[2].get_xlim()
    axes[2].plot([x0, x1], [x0, x1], "k--", label="Expected")

    fig.tight_layout()
    if save:
        out_png = BASE_DIR / (
            f"independent_signals_{cfg.output_suffix}"
            f"_fdr{str(FDR_ALPHA).replace('.', 'p')}"
            f"_margfrac{str(MIN_FRAC_CELLS_MARG_SIG).replace('.', 'p')}"
            f"_discfrac{str(DISCOVERY_MIN_FRAC_CELLLEVEL).replace('.', 'p')}.png"
        )
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        print(f"Saved -> {out_png}")
    return fig


def build_independent_method_comparison(
    comparison: MethodComparisonConfig,
    results_by_key: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    parts: list[pd.DataFrame] = []
    for member in comparison.members:
        df = independent_results_for_plot(results_by_key.get(member.cfg_key, pd.DataFrame()))
        if df.empty or "method" not in df.columns:
            continue
        sub = df[df["method"] == member.source_method].copy()
        if sub.empty:
            continue
        sub["method"] = member.plot_method
        sub["method_label"] = member.label
        sub["comparison"] = comparison.key
        parts.append(sub)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def _plot_independent_comparison_metric(
    ax: plt.Axes,
    df: pd.DataFrame,
    *,
    x_col: str,
    y_col: str,
    title: str,
    ylabel: str,
    method_order: Sequence[str],
    method_name_map: dict[str, str],
    ylim: Optional[tuple[float, float]] = None,
) -> None:
    _plot_mean_ci_lines_by_method(
        ax,
        df,
        x_col=x_col,
        y_col=y_col,
        title=title,
        xlabel=SIM_BY_KEY[DENOISING_COMPARISON.base_cfg_key].indep_x_label,
        ylabel=ylabel,
        method_order=method_order,
        method_name_map=method_name_map,
        ylim=ylim,
        legend=True,
    )


def plot_independent_method_comparison_figure(
    df: pd.DataFrame,
    comparison: MethodComparisonConfig,
    *,
    save: bool = True,
) -> plt.Figure:
    if df.empty:
        raise ValueError(f"{comparison.title}: no independent-signal comparison data to plot.")

    base_cfg = SIM_BY_KEY[comparison.base_cfg_key]
    x_col = base_cfg.indep_x_col
    fig, axes = plt.subplots(1, 3, figsize=(INDEP_FIG_W, INDEP_FIG_H))
    fig.suptitle(comparison.title, fontsize=FONT_SIZE_TITLE + 2, y=1.02)

    _plot_independent_comparison_metric(
        axes[0],
        df,
        x_col=x_col,
        y_col="n_true_indep_signals_selected",
        title="True independent signals detected",
        ylabel="True indep. signals",
        method_order=comparison.method_order,
        method_name_map=comparison.method_name_map,
    )
    _plot_independent_comparison_metric(
        axes[1],
        df,
        x_col=x_col,
        y_col="n_false_indep_signals_selected",
        title="False independent signals detected",
        ylabel="False indep. signals",
        method_order=comparison.method_order,
        method_name_map=comparison.method_name_map,
    )
    _plot_independent_comparison_metric(
        axes[2],
        df,
        x_col=x_col,
        y_col="metacell_self_agreement_majority_leiden",
        title="Signal self agreement",
        ylabel="Within signal purity",
        method_order=comparison.method_order,
        method_name_map=comparison.method_name_map,
        ylim=(0.0, 1.0),
    )

    fig.tight_layout()
    if save:
        out_png = BASE_DIR / (
            f"independent_signals_{comparison.output_suffix}"
            f"_fdr{str(FDR_ALPHA).replace('.', 'p')}"
            f"_margfrac{str(MIN_FRAC_CELLS_MARG_SIG).replace('.', 'p')}"
            f"_discfrac{str(DISCOVERY_MIN_FRAC_CELLLEVEL).replace('.', 'p')}.png"
        )
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        print(f"Saved -> {out_png}")
    return fig


In [14]:
independent_comparison_results: dict[str, pd.DataFrame] = {}

for cfg in SIMULATION_SETS:
    print(f"Plotting independent-signal summary: {cfg.title}")
    display(independent_results[cfg.key].head())
    if independent_results[cfg.key].empty:
        print(f"Skipping {cfg.title}: no independent-signal rows were evaluated.")
        continue
    plot_independent_signal_figure(independent_results[cfg.key], cfg)
    plt.show()

for comparison in METHOD_COMPARISONS:
    print(f"Plotting independent-signal comparison: {comparison.title}")
    comp_df = build_independent_method_comparison(comparison, independent_results)
    independent_comparison_results[comparison.key] = comp_df
    if comp_df.empty:
        print(f"Skipping {comparison.title}: no comparison rows were available.")
        continue
    display(comp_df.head())
    plot_independent_method_comparison_figure(comp_df, comparison)
    plt.show()


Plotting independent-signal summary: Original simulations


,simulation_set,cluster,replicate,src_n,causal_clusters,overlap_k,causal_genes_per_cluster,geneset_file,geneset_path,method,...,has_neg1_indep_signal,indep_root,n_indep_signals_selected,metacell_self_agreement_majority_leiden,n_false_indep_signals_selected,n_leiden_clusters_discovered_in_truth,n_true_indep_signals_selected,n_metacells_selected_total,n_indep_signals_selected_int,n_leiden_clusters_discovered_in_truth_int
0,original,0,0,1,1,50,50,TMS_FACS_0_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,1.0,0.936842,0.0,1.0,1.0,95,1.0,1.0
1,original,0,1,1,1,50,50,TMS_FACS_0_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,2.0,0.953488,0.0,1.0,2.0,86,2.0,1.0
2,original,0,2,1,1,50,50,TMS_FACS_0_2_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,3.0,0.940594,1.0,1.0,2.0,101,3.0,1.0
3,original,1,0,1,1,50,50,TMS_FACS_1_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,1.0,0.988636,0.0,1.0,1.0,88,1.0,1.0
4,original,1,1,1,1,50,50,TMS_FACS_1_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,1.0,0.958333,0.0,1.0,1.0,72,1.0,1.0


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/independent_signals_original_fdr0p1_margfrac0p05_discfrac0p05.png
Plotting independent-signal summary: Cell-percent simulations


,simulation_set,cluster,replicate,src_n,causal_clusters,overlap_k,causal_genes_per_cluster,geneset_file,geneset_path,method,...,indep_root,min_cell_percent,n_indep_signals_selected,metacell_self_agreement_majority_leiden,n_false_indep_signals_selected,n_leiden_clusters_discovered_in_truth,n_true_indep_signals_selected,n_metacells_selected_total,n_indep_signals_selected_int,n_leiden_clusters_discovered_in_truth_int
0,sims_percent,0,0,3,3,50,50,TMS_FACS_0_0_pct1_ov50_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,/workspace/scdrsfm_local/results/sim/simulatio...,1,4.0,0.880952,1.0,3.0,3.0,168,4.0,3.0
1,sims_percent,0,1,3,3,50,50,TMS_FACS_0_1_pct1_ov50_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,/workspace/scdrsfm_local/results/sim/simulatio...,1,3.0,0.824324,0.0,3.0,3.0,148,3.0,3.0
2,sims_percent,0,2,3,3,50,50,TMS_FACS_0_2_pct1_ov50_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,/workspace/scdrsfm_local/results/sim/simulatio...,1,3.0,0.855856,0.0,3.0,3.0,111,3.0,3.0
3,sims_percent,1,0,3,3,50,50,TMS_FACS_1_0_pct1_ov50_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,/workspace/scdrsfm_local/results/sim/simulatio...,1,4.0,0.924370,0.0,3.0,4.0,119,4.0,3.0
4,sims_percent,1,1,3,3,50,50,TMS_FACS_1_1_pct1_ov50_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,/workspace/scdrsfm_local/results/sim/simulatio...,1,6.0,1.000000,1.0,3.0,5.0,58,6.0,3.0


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/independent_signals_sims_percent_fdr0p1_margfrac0p05_discfrac0p05.png
Plotting independent-signal summary: Causal-gene simulations


,simulation_set,cluster,replicate,src_n,causal_clusters,overlap_k,causal_genes_per_cluster,geneset_file,geneset_path,method,...,has_neg1_indep_signal,indep_root,n_indep_signals_selected,metacell_self_agreement_majority_leiden,n_false_indep_signals_selected,n_leiden_clusters_discovered_in_truth,n_true_indep_signals_selected,n_metacells_selected_total,n_indep_signals_selected_int,n_leiden_clusters_discovered_in_truth_int
0,sims_genes,0,0,3,3,25,25,TMS_FACS_0_0_ov25_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,0.0,NaN,0.0,0.0,0.0,0,0.0,0.0
1,sims_genes,0,1,3,3,25,25,TMS_FACS_0_1_ov25_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,2.0,0.931034,0.0,2.0,2.0,58,2.0,2.0
2,sims_genes,0,2,3,3,25,25,TMS_FACS_0_2_ov25_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,1.0,0.573529,0.0,2.0,1.0,68,1.0,2.0
3,sims_genes,1,0,3,3,25,25,TMS_FACS_1_0_ov25_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,3.0,0.982143,0.0,2.0,3.0,56,3.0,2.0
4,sims_genes,1,1,3,3,25,25,TMS_FACS_1_1_ov25_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,scdrs+_conditional,...,False,/workspace/scdrsfm_local/results/sim/simulatio...,2.0,0.913043,0.0,2.0,2.0,23,2.0,2.0


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/independent_signals_sims_genes_fdr0p1_margfrac0p05_discfrac0p05.png
Plotting independent-signal comparison: Denoising comparison: MAGIC vs KNN


,simulation_set,cluster,replicate,src_n,causal_clusters,overlap_k,causal_genes_per_cluster,geneset_file,geneset_path,method,...,n_indep_signals_selected,metacell_self_agreement_majority_leiden,n_false_indep_signals_selected,n_leiden_clusters_discovered_in_truth,n_true_indep_signals_selected,n_metacells_selected_total,n_indep_signals_selected_int,n_leiden_clusters_discovered_in_truth_int,method_label,comparison
0,original,0,0,1,1,50,50,TMS_FACS_0_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,magic,...,1.0,0.936842,0.0,1.0,1.0,95,1.0,1.0,MAGIC,magic_vs_knn
1,original,0,1,1,1,50,50,TMS_FACS_0_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,magic,...,2.0,0.953488,0.0,1.0,2.0,86,2.0,1.0,MAGIC,magic_vs_knn
2,original,0,2,1,1,50,50,TMS_FACS_0_2_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,magic,...,3.0,0.940594,1.0,1.0,2.0,101,3.0,1.0,MAGIC,magic_vs_knn
3,original,1,0,1,1,50,50,TMS_FACS_1_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,magic,...,1.0,0.988636,0.0,1.0,1.0,88,1.0,1.0,MAGIC,magic_vs_knn
4,original,1,1,1,1,50,50,TMS_FACS_1_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,magic,...,1.0,0.958333,0.0,1.0,1.0,72,1.0,1.0,MAGIC,magic_vs_knn


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/independent_signals_magic_vs_knn_fdr0p1_margfrac0p05_discfrac0p05.png


## Independent-signal precision/recall

This section computes precision/recall for scDRS-FM independent-signal variants and adds two scDRS baselines:

- **scDRS**: all marginally associated cells are treated as one signal.
- **scDRS w/ Leiden**: marginally associated cells are split by a fresh Leiden clustering at resolution 1.0, and a Leiden cluster is counted as an inferred signal when at least 5% of its cells are associated.


In [15]:
def _parse_semicolon_set(s: Any) -> set[str]:
    if s is None:
        return set()
    return {tok.strip() for tok in str(s).split(";") if tok.strip() and tok.strip().lower() != "nan"}


def _agg_mean_ci_simple(df: pd.DataFrame, x_col: str, y_col: str) -> pd.DataFrame:
    d = df.dropna(subset=[x_col, y_col]).copy()
    rows = []
    for xval, group in d.groupby(x_col, sort=True):
        mean, ci_half, n = _mean_ci95_t(group[y_col].to_numpy(dtype=np.float64))
        rows.append({"x": float(xval), "mean": mean, "ci_half": ci_half, "n": int(n)})
    return pd.DataFrame(rows).sort_values("x").reset_index(drop=True) if rows else pd.DataFrame()


def _agg_mean_ci_by_signal(df: pd.DataFrame, signal_col_name: str, x_col: str, y_col: str) -> pd.DataFrame:
    parts = []
    for signal_name, group in df.groupby(signal_col_name, sort=False):
        agg = _agg_mean_ci_simple(group, x_col, y_col)
        if agg.empty:
            continue
        agg[signal_col_name] = signal_name
        parts.append(agg)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["x", "mean", "ci_half", "n", signal_col_name])


def _build_cell_to_signal(df_sel: pd.DataFrame, signal_col: str) -> dict[str, int]:
    out: dict[str, int] = {}
    if df_sel is None or df_sel.empty or signal_col not in df_sel.columns:
        return out

    for _, row in df_sel.iterrows():
        sig_val = pd.to_numeric(row.get(signal_col), errors="coerce")
        if pd.isna(sig_val) or int(sig_val) < 0:
            continue
        cids = str(row.get("cell_ids", ""))
        if not cids or cids.lower() == "nan":
            continue
        for cell in cids.split(","):
            cell = cell.strip()
            if cell:
                out.setdefault(cell, int(sig_val))
    return out


def _build_signal_to_cells(cell_to_signal: dict[str, Any]) -> dict[Any, set[str]]:
    out: dict[Any, set[str]] = {}
    for cell, sig in cell_to_signal.items():
        out.setdefault(sig, set()).add(str(cell))
    return out


def _signal_to_matched_causal_labels(
    *,
    signal_to_cells: dict[Any, set[str]],
    causal_labels: set[str],
    cell_to_label: dict[str, str],
    total_counts: dict[str, int],
    min_frac_of_causal: float,
) -> dict[Any, set[str]]:
    causal_labels = {str(x) for x in causal_labels}
    signal_matches: dict[Any, set[str]] = {}

    for sig, cells in signal_to_cells.items():
        hit_counts: dict[str, int] = {}
        for cell in cells:
            lab = cell_to_label.get(cell)
            if lab is not None and str(lab) in causal_labels:
                hit_counts[str(lab)] = hit_counts.get(str(lab), 0) + 1

        matched: set[str] = set()
        for lab in causal_labels:
            denom = int(total_counts.get(lab, 0))
            if denom > 0 and (hit_counts.get(lab, 0) / float(denom)) > float(min_frac_of_causal):
                matched.add(lab)
        signal_matches[sig] = matched
    return signal_matches


def _signal_top_causal_label_by_proportion(
    *,
    signal_to_cells: dict[Any, set[str]],
    causal_labels: set[str],
    cell_to_label: dict[str, str],
) -> dict[Any, dict[str, Any]]:
    causal_labels = {str(x) for x in causal_labels}
    out: dict[Any, dict[str, Any]] = {}

    for sig, cells in signal_to_cells.items():
        causal_counts: dict[str, int] = {}
        signal_size = 0
        for cell in cells:
            lab = cell_to_label.get(cell)
            if lab is None:
                continue
            signal_size += 1
            lab = str(lab)
            if lab in causal_labels:
                causal_counts[lab] = causal_counts.get(lab, 0) + 1

        if signal_size == 0 or not causal_counts:
            out[sig] = {"top_label": None, "top_count": 0, "signal_size": signal_size}
            continue

        top_label, top_count = max(causal_counts.items(), key=lambda kv: (kv[1], kv[0]))
        out[sig] = {"top_label": str(top_label), "top_count": int(top_count), "signal_size": signal_size}
    return out


def _precision_recall_from_signal_to_cells(
    *,
    signal_to_cells: dict[Any, set[str]],
    causal_labels: set[str],
    cell_to_label: dict[str, str],
    total_counts: dict[str, int],
    min_frac_of_causal: float,
) -> dict[str, float]:
    causal_set = {str(x) for x in causal_labels}
    n_true = int(len(causal_set))
    signal_to_cells = {sig: set(cells) for sig, cells in signal_to_cells.items() if cells}
    inferred_signals = set(signal_to_cells.keys())
    n_inferred = int(len(inferred_signals))

    if n_inferred == 0:
        return {
            "n_true_causal_clusters": float(n_true),
            "precision": np.nan,
            "recall": 0.0,
            "n_inferred_populations": 0.0,
            "precision_credit_sum": 0.0,
            "n_recalled_causal_populations": 0.0,
        }

    signal_to_matched = _signal_to_matched_causal_labels(
        signal_to_cells=signal_to_cells,
        causal_labels=causal_set,
        cell_to_label=cell_to_label,
        total_counts=total_counts,
        min_frac_of_causal=min_frac_of_causal,
    )
    precision_credit_by_signal = {
        sig: (1.0 / len(signal_to_matched.get(sig, set()))) if len(signal_to_matched.get(sig, set())) > 0 else 0.0
        for sig in inferred_signals
    }
    precision_credit_sum = float(sum(precision_credit_by_signal.values()))
    precision = precision_credit_sum / float(n_inferred) if n_inferred else np.nan

    signal_top_info = _signal_top_causal_label_by_proportion(
        signal_to_cells=signal_to_cells,
        causal_labels=causal_set,
        cell_to_label=cell_to_label,
    )
    recalled_causal_labels: set[str] = set()
    for info in signal_top_info.values():
        top_label = info["top_label"]
        top_count = int(info["top_count"])
        if top_label is None:
            continue
        denom = int(total_counts.get(str(top_label), 0))
        if denom > 0 and (top_count / float(denom)) > float(min_frac_of_causal):
            recalled_causal_labels.add(str(top_label))

    n_recalled = int(len(recalled_causal_labels))
    recall = n_recalled / float(n_true) if n_true else np.nan
    return {
        "n_true_causal_clusters": float(n_true),
        "precision": float(precision) if np.isfinite(precision) else np.nan,
        "recall": float(recall) if np.isfinite(recall) else np.nan,
        "n_inferred_populations": float(n_inferred),
        "precision_credit_sum": float(precision_credit_sum),
        "n_recalled_causal_populations": float(n_recalled),
    }


def _base_pr_row_from_run(cfg: SimulationSetConfig, run: pd.Series) -> dict[str, Any]:
    keep_cols = [
        "simulation_set",
        "cluster",
        "replicate",
        "src_n",
        "causal_clusters",
        "overlap_k",
        "causal_genes_per_cluster",
        "min_cell_percent",
        cfg.indep_x_col,
        "geneset_file",
        "geneset_path",
        "causal_leiden_labels",
    ]
    row = {col: run.get(col, np.nan) for col in keep_cols if col in run.index or col == cfg.indep_x_col}
    row["simulation_set"] = cfg.key
    if cfg.indep_x_col not in row:
        row[cfg.indep_x_col] = run.get(cfg.indep_x_col, np.nan)
    return row


def compute_fm_signal_precision_recall_for_set(
    cfg: SimulationSetConfig,
    indep_df: pd.DataFrame,
    *,
    obs_names: pd.Index,
    cell_to_leiden: dict[str, str],
    total_leiden_counts: dict[str, int],
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    df_runs = independent_results_for_plot(indep_df)
    df_runs = df_runs[df_runs["method"] == "scdrs+_conditional"].copy()

    for _, run in df_runs.iterrows():
        cond_path = Path(run["conditional_file"])
        marg_path = Path(run["marginal_file"])
        if not cond_path.exists() or not marg_path.exists():
            continue

        causal_set = _parse_semicolon_set(run.get("causal_leiden_labels", ""))
        n_true = int(len(causal_set))
        if n_true == 0:
            continue

        try:
            sig_cells_marg, _ = _read_marginal(marg_path, obs_names, alpha=FDR_ALPHA)
            sig_cells_marg_set = set(sig_cells_marg.astype(str).tolist())
            df_cond = pd.read_csv(cond_path, sep="\t", compression="gzip", index_col=0)
        except Exception:
            continue

        if "cell_ids" not in df_cond.columns:
            continue
        available_signal_cols = ordered_signal_cols([col for col in SIGNAL_COLS if col in df_cond.columns])
        if not available_signal_cols:
            continue

        try:
            pcol = _pick_first_existing_col(df_cond, COND_PVAL_COL_CANDIDATES, what="conditional p-value")
        except Exception:
            continue

        df_base, _ = select_metacells_by_pvalue_and_marginal(
            df_cond,
            pcol=pcol,
            sig_cells_marg=sig_cells_marg_set,
            fdr_alpha=FDR_ALPHA,
            min_frac_cells_marg_sig=MIN_FRAC_CELLS_MARG_SIG,
        )

        for signal_col in available_signal_cols:
            row_base = _base_pr_row_from_run(cfg, run)
            row_base.update({
                "method": "scdrs+_conditional",
                "signal_col": signal_col,
                "default_signal_col": choose_default_signal_col(available_signal_cols),
            })

            df_sel = restrict_to_assigned_signal(df_base, signal_col)
            if df_sel is None or df_sel.empty:
                metrics = _precision_recall_from_signal_to_cells(
                    signal_to_cells={},
                    causal_labels=causal_set,
                    cell_to_label=cell_to_leiden,
                    total_counts=total_leiden_counts,
                    min_frac_of_causal=MATCH_MIN_FRAC_OF_CAUSAL,
                )
                rows.append({**row_base, **metrics})
                continue

            cond_cells = _cells_in_selected_metacells(df_sel)
            inter_cells = cond_cells.intersection(sig_cells_marg_set)
            cell_to_signal_all = _build_cell_to_signal(df_sel, signal_col)
            cell_to_signal = {cell: sig for cell, sig in cell_to_signal_all.items() if cell in inter_cells}
            signal_to_cells = _build_signal_to_cells(cell_to_signal)

            metrics = _precision_recall_from_signal_to_cells(
                signal_to_cells=signal_to_cells,
                causal_labels=causal_set,
                cell_to_label=cell_to_leiden,
                total_counts=total_leiden_counts,
                min_frac_of_causal=MATCH_MIN_FRAC_OF_CAUSAL,
            )
            rows.append({**row_base, **metrics})

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    if cfg.indep_x_col in out.columns:
        out[cfg.indep_x_col] = pd.to_numeric(out[cfg.indep_x_col], errors="coerce")
    out["n_true_causal_clusters"] = (
        pd.to_numeric(out["n_true_causal_clusters"], errors="coerce").round().astype("Int64").astype(float)
    )
    return out


def _metadata_from_geneset_for_pr(
    cfg: SimulationSetConfig,
    pred_df: pd.DataFrame,
    gs_path: Path,
) -> Optional[dict[str, Any]]:
    try:
        meta = parse_geneset_filename(gs_path, cfg)
        trait = _read_trait_from_gs(gs_path)
        trait_info = _parse_trait_info(trait)
    except Exception:
        return None

    causal_labels = _get_causal_labels_from_predictions(
        pred_df,
        cfg,
        gs_path=gs_path,
        meta=meta,
        trait=trait,
    )
    if causal_labels is None or len(causal_labels) == 0:
        causal_labels = set(trait_info["donors"])
    if not causal_labels:
        return None

    row: dict[str, Any] = {
        "simulation_set": cfg.key,
        "cluster": int(meta["cluster"]),
        "replicate": int(meta["replicate"]),
        "src_n": int(meta["src_n"]),
        "causal_clusters": int(meta["src_n"]),
        "overlap_k": int(meta["overlap_k"]),
        "causal_genes_per_cluster": int(meta["overlap_k"]),
        "geneset_file": gs_path.name,
        "geneset_path": str(gs_path),
        "causal_leiden_labels": ";".join(sorted({str(x) for x in causal_labels})),
    }
    if "min_cell_percent" in meta:
        row["min_cell_percent"] = int(meta["min_cell_percent"])
    if cfg.indep_x_col in row:
        pass
    elif cfg.indep_x_col in meta:
        row[cfg.indep_x_col] = meta[cfg.indep_x_col]

    return {
        "meta": meta,
        "trait": trait,
        "trait_info": trait_info,
        "causal_labels": {str(x) for x in causal_labels},
        "row": row,
    }


def _scdrs_leiden_signal_to_cells(
    sig_cells: set[str],
    *,
    cell_to_cluster: dict[str, str],
    cluster_total_counts: dict[str, int],
    min_frac_sig: float,
) -> dict[int, set[str]]:
    sig_counts: dict[str, int] = {}
    for cell in sig_cells:
        cluster = cell_to_cluster.get(cell)
        if cluster is not None:
            sig_counts[str(cluster)] = sig_counts.get(str(cluster), 0) + 1

    selected_clusters = []
    for cluster, n_sig in sig_counts.items():
        denom = int(cluster_total_counts.get(str(cluster), 0))
        if denom > 0 and (n_sig / float(denom)) >= float(min_frac_sig):
            selected_clusters.append(str(cluster))
    selected_clusters = sorted(selected_clusters)
    cluster_to_signal = {cluster: i for i, cluster in enumerate(selected_clusters)}

    out: dict[int, set[str]] = {cluster_to_signal[cluster]: set() for cluster in selected_clusters}
    for cell in sig_cells:
        cluster = cell_to_cluster.get(cell)
        if cluster is None:
            continue
        cluster = str(cluster)
        if cluster in cluster_to_signal:
            out[cluster_to_signal[cluster]].add(str(cell))
    return out


def compute_scdrs_marginal_precision_recall_for_set(
    cfg: SimulationSetConfig,
    *,
    obs_names: pd.Index,
    cell_to_leiden: dict[str, str],
    total_leiden_counts: dict[str, int],
    cell_to_scdrs_leiden: dict[str, str],
    total_scdrs_leiden_counts: dict[str, int],
    use_leiden: bool,
) -> pd.DataFrame:
    method = "scdrs_marginal_leiden" if use_leiden else "scdrs_marginal"
    signal_col = method
    pred_df = load_predictions(cfg)
    rows: list[dict[str, Any]] = []

    for gs_path in geneset_files(cfg):
        run_info = _metadata_from_geneset_for_pr(cfg, pred_df, gs_path)
        if run_info is None:
            continue
        meta = run_info["meta"]
        trait = run_info["trait"]
        causal_set = run_info["causal_labels"]
        row_base = run_info["row"]

        prefixes = [trait, gs_path.name, gs_path.stem]
        scdrs_out = result_dir_for_run(cfg.scdrs_root, cfg, meta)
        marg_path = _find_marginal_file(scdrs_out, prefixes)
        if marg_path is None:
            continue

        try:
            sig_cells_marg, _ = _read_marginal(marg_path, obs_names, alpha=FDR_ALPHA)
        except Exception:
            continue

        sig_cells = set(sig_cells_marg.astype(str).tolist())
        if use_leiden:
            signal_to_cells = _scdrs_leiden_signal_to_cells(
                sig_cells,
                cell_to_cluster=cell_to_scdrs_leiden,
                cluster_total_counts=total_scdrs_leiden_counts,
                min_frac_sig=SCDRS_LEIDEN_MIN_FRAC_SIG,
            )
        else:
            signal_to_cells = {0: sig_cells} if sig_cells else {}

        metrics = _precision_recall_from_signal_to_cells(
            signal_to_cells=signal_to_cells,
            causal_labels=causal_set,
            cell_to_label=cell_to_leiden,
            total_counts=total_leiden_counts,
            min_frac_of_causal=MATCH_MIN_FRAC_OF_CAUSAL,
        )
        rows.append({
            **row_base,
            "method": method,
            "signal_col": signal_col,
            "default_signal_col": signal_col,
            "marginal_file": str(marg_path),
            "n_sig_cells": float(len(sig_cells)),
            "scdrs_leiden_resolution": SCDRS_LEIDEN_RESOLUTION if use_leiden else np.nan,
            "scdrs_leiden_min_frac_sig": SCDRS_LEIDEN_MIN_FRAC_SIG if use_leiden else np.nan,
            **metrics,
        })

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    if cfg.indep_x_col in out.columns:
        out[cfg.indep_x_col] = pd.to_numeric(out[cfg.indep_x_col], errors="coerce")
    out["n_true_causal_clusters"] = (
        pd.to_numeric(out["n_true_causal_clusters"], errors="coerce").round().astype("Int64").astype(float)
    )
    return out


def compute_signal_precision_recall_for_set(
    cfg: SimulationSetConfig,
    indep_df: pd.DataFrame,
    *,
    obs_names: pd.Index,
    cell_to_leiden: dict[str, str],
    total_leiden_counts: dict[str, int],
    cell_to_scdrs_leiden: Optional[dict[str, str]] = None,
    total_scdrs_leiden_counts: Optional[dict[str, int]] = None,
    include_scdrs_baselines: bool = True,
) -> pd.DataFrame:
    parts: list[pd.DataFrame] = [
        compute_fm_signal_precision_recall_for_set(
            cfg,
            indep_df,
            obs_names=obs_names,
            cell_to_leiden=cell_to_leiden,
            total_leiden_counts=total_leiden_counts,
        )
    ]

    if include_scdrs_baselines:
        if cell_to_scdrs_leiden is None or total_scdrs_leiden_counts is None:
            raise ValueError("scDRS Leiden labels are required when include_scdrs_baselines=True")
        """
        parts.append(
            compute_scdrs_marginal_precision_recall_for_set(
                cfg,
                obs_names=obs_names,
                cell_to_leiden=cell_to_leiden,
                total_leiden_counts=total_leiden_counts,
                cell_to_scdrs_leiden=cell_to_scdrs_leiden,
                total_scdrs_leiden_counts=total_scdrs_leiden_counts,
                use_leiden=False,
            )
        )
        """
        parts.append(
            compute_scdrs_marginal_precision_recall_for_set(
                cfg,
                obs_names=obs_names,
                cell_to_leiden=cell_to_leiden,
                total_leiden_counts=total_leiden_counts,
                cell_to_scdrs_leiden=cell_to_scdrs_leiden,
                total_scdrs_leiden_counts=total_scdrs_leiden_counts,
                use_leiden=True,
            )
        )

    parts = [part for part in parts if part is not None and not part.empty]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


In [16]:
PR_FIG_SIZE_W = 8
PR_FIG_SIZE_H = 8
PR_FONT_SIZE_LABEL = 38
PR_FONT_SIZE_TICK = 26
PR_FONT_SIZE_LEGEND = 24
PR_CAPSIZE = 4
PR_ERR_LINEWIDTH = 2.0


def _aligned_one_signal(agg: pd.DataFrame, signal_col: str, x_values: Sequence[float]) -> tuple[np.ndarray, np.ndarray]:
    sub = agg[agg["signal_col"] == signal_col].copy()
    sub = sub.set_index("x").reindex(x_values).reset_index()
    return sub["mean"].to_numpy(dtype=float), sub["ci_half"].to_numpy(dtype=float)


def _agg_mean_ci_by_method_for_pr(df: pd.DataFrame, *, x_col: str, y_col: str) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    d = df.dropna(subset=[x_col, y_col]).copy()
    for (method, xval), group in d.groupby(["method", x_col], sort=True):
        mean, ci_half, n = _mean_ci95_t(group[y_col].to_numpy(dtype=np.float64))
        rows.append({"method": method, "x": float(xval), "mean": mean, "ci_half": ci_half, "n": int(n)})
    return pd.DataFrame(rows).sort_values(["method", "x"]).reset_index(drop=True) if rows else pd.DataFrame()


def _aligned_one_method(agg: pd.DataFrame, method: str, x_values: Sequence[float]) -> tuple[np.ndarray, np.ndarray]:
    sub = agg[agg["method"] == method].copy()
    sub = sub.set_index("x").reindex(x_values).reset_index()
    return sub["mean"].to_numpy(dtype=float), sub["ci_half"].to_numpy(dtype=float)


def plot_signal_variant_precision_recall_figure(plot_long: pd.DataFrame, cfg: SimulationSetConfig, *, save: bool = True) -> plt.Figure:
    # Plot all scDRS-FM independent-signal variants for one simulation set.
    plot_long = plot_long[plot_long["method"] == "scdrs+_conditional"].copy()
    if plot_long.empty:
        raise ValueError(f"{cfg.title}: no scDRS-FM precision/recall data to plot.")

    x_col = cfg.indep_x_col
    default_signal_col = choose_default_signal_col(plot_long["signal_col"].dropna().unique())
    prec_agg = _agg_mean_ci_by_signal(plot_long, "signal_col", x_col, "precision")
    rec_agg = _agg_mean_ci_by_signal(plot_long, "signal_col", x_col, "recall")
    if prec_agg.empty and rec_agg.empty:
        raise ValueError(f"{cfg.title}: precision/recall aggregations are empty.")

    x_values = sorted(pd.unique(plot_long[x_col].dropna()).astype(float))
    signals_in_data = ordered_signal_cols(plot_long["signal_col"].dropna().unique())

    fig, axes = plt.subplots(1, 2, figsize=(2 * PR_FIG_SIZE_W, PR_FIG_SIZE_H))
    fig.suptitle(f"{cfg.title}: scDRS-FM signal variants", fontsize=PR_FONT_SIZE_LABEL + 2, y=1.03)
    ax_prec, ax_rec = axes

    ax_prec.tick_params(axis="both", labelsize=PR_FONT_SIZE_TICK)
    ax_prec.set_ylim(0.0, 1.01)
    for signal_col in signals_in_data:
        if prec_agg[prec_agg["signal_col"] == signal_col].empty:
            continue
        y, yerr = _aligned_one_signal(prec_agg, signal_col, x_values)
        line, = ax_prec.plot(
            x_values,
            y,
            marker="o",
            linewidth=LINEWIDTH,
            label=signal_display_name(signal_col, default_signal_col),
        )
        ax_prec.errorbar(x_values, y, yerr=yerr, fmt="none", capsize=PR_CAPSIZE, linewidth=PR_ERR_LINEWIDTH, color=line.get_color())
    ax_prec.set_ylabel("Precision", fontsize=PR_FONT_SIZE_LABEL)
    ax_prec.set_xlabel(cfg.indep_x_label, fontsize=PR_FONT_SIZE_LABEL)
    ax_prec.set_xticks(x_values)
    ax_prec.set_xticklabels([str(int(x)) if float(x).is_integer() else str(x) for x in x_values])

    ax_rec.tick_params(axis="both", labelsize=PR_FONT_SIZE_TICK)
    ax_rec.set_ylim(0.0, 1.01)
    for signal_col in signals_in_data:
        if rec_agg[rec_agg["signal_col"] == signal_col].empty:
            continue
        y, yerr = _aligned_one_signal(rec_agg, signal_col, x_values)
        line, = ax_rec.plot(
            x_values,
            y,
            marker="o",
            linewidth=LINEWIDTH,
            label=signal_display_name(signal_col, default_signal_col),
        )
        ax_rec.errorbar(x_values, y, yerr=yerr, fmt="none", capsize=PR_CAPSIZE, linewidth=PR_ERR_LINEWIDTH, color=line.get_color())
    ax_rec.set_ylabel("Recall", fontsize=PR_FONT_SIZE_LABEL)
    ax_rec.set_xlabel(cfg.indep_x_label, fontsize=PR_FONT_SIZE_LABEL)
    ax_rec.set_xticks(x_values)
    ax_rec.set_xticklabels([str(int(x)) if float(x).is_integer() else str(x) for x in x_values])
    ax_rec.legend(fontsize=PR_FONT_SIZE_LEGEND, frameon=True, loc="best")

    fig.tight_layout()
    if save:
        out_png = BASE_DIR / f"all_signals_precision_recall_{cfg.output_suffix}.png"
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        print(f"Saved -> {out_png}")
    return fig


def _default_method_pr_subset(plot_long: pd.DataFrame) -> pd.DataFrame:
    fm = plot_long[plot_long["method"] == "scdrs+_conditional"].copy()
    default_signal_col = choose_default_signal_col(fm["signal_col"].dropna().unique()) if not fm.empty else None
    parts: list[pd.DataFrame] = []
    if default_signal_col is not None:
        parts.append(fm[fm["signal_col"] == default_signal_col].copy())
    for method in ["scdrs_marginal", "scdrs_marginal_leiden"]:
        sub = plot_long[plot_long["method"] == method].copy()
        if not sub.empty:
            parts.append(sub)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def plot_pr_methods_figure(
    plot_long: pd.DataFrame,
    cfg: SimulationSetConfig,
    *,
    title: str,
    output_suffix: str,
    method_order: Sequence[str],
    method_name_map: Optional[dict[str, str]] = None,
    save: bool = True,
) -> plt.Figure:
    if plot_long.empty:
        raise ValueError(f"{title}: no precision/recall data to plot.")

    label_map = {**METHOD_NAME_MAP, **(method_name_map or {})}
    x_col = cfg.indep_x_col
    prec_agg = _agg_mean_ci_by_method_for_pr(plot_long, x_col=x_col, y_col="precision")
    rec_agg = _agg_mean_ci_by_method_for_pr(plot_long, x_col=x_col, y_col="recall")
    if prec_agg.empty and rec_agg.empty:
        raise ValueError(f"{title}: precision/recall aggregations are empty.")

    x_values = sorted(pd.unique(plot_long[x_col].dropna()).astype(float))
    methods_present = [method for method in method_order if (plot_long["method"] == method).any()]

    fig, axes = plt.subplots(1, 2, figsize=(2 * PR_FIG_SIZE_W, PR_FIG_SIZE_H))
    fig.suptitle(title, fontsize=PR_FONT_SIZE_LABEL + 2, y=1.03)
    ax_prec, ax_rec = axes

    ax_prec.tick_params(axis="both", labelsize=PR_FONT_SIZE_TICK)
    ax_prec.set_ylim(0.0, 1.01)
    for method in methods_present:
        if prec_agg[prec_agg["method"] == method].empty:
            continue
        y, yerr = _aligned_one_method(prec_agg, method, x_values)
        line, = ax_prec.plot(x_values, y, marker="o", linewidth=LINEWIDTH, label=label_map.get(method, method))
        ax_prec.errorbar(x_values, y, yerr=yerr, fmt="none", capsize=PR_CAPSIZE, linewidth=PR_ERR_LINEWIDTH, color=line.get_color())
    ax_prec.set_ylabel("Precision", fontsize=PR_FONT_SIZE_LABEL)
    ax_prec.set_xlabel(cfg.indep_x_label, fontsize=PR_FONT_SIZE_LABEL)
    ax_prec.set_xticks(x_values)
    ax_prec.set_xticklabels([str(int(x)) if float(x).is_integer() else str(x) for x in x_values])

    ax_rec.tick_params(axis="both", labelsize=PR_FONT_SIZE_TICK)
    ax_rec.set_ylim(0.0, 1.01)
    for method in methods_present:
        if rec_agg[rec_agg["method"] == method].empty:
            continue
        y, yerr = _aligned_one_method(rec_agg, method, x_values)
        line, = ax_rec.plot(x_values, y, marker="o", linewidth=LINEWIDTH, label=label_map.get(method, method))
        ax_rec.errorbar(x_values, y, yerr=yerr, fmt="none", capsize=PR_CAPSIZE, linewidth=PR_ERR_LINEWIDTH, color=line.get_color())
    ax_rec.set_ylabel("Recall", fontsize=PR_FONT_SIZE_LABEL)
    ax_rec.set_xlabel(cfg.indep_x_label, fontsize=PR_FONT_SIZE_LABEL)
    ax_rec.set_xticks(x_values)
    ax_rec.set_xticklabels([str(int(x)) if float(x).is_integer() else str(x) for x in x_values])
    #ax_rec.legend(fontsize=PR_FONT_SIZE_LEGEND, frameon=True, loc="best")

    fig.tight_layout()
    if save:
        out_png = BASE_DIR / f"precision_recall_{output_suffix}.png"
        fig.savefig(out_png, dpi=300, bbox_inches="tight")
        print(f"Saved -> {out_png}")
    return fig


def plot_default_method_precision_recall_figure(plot_long: pd.DataFrame, cfg: SimulationSetConfig, *, save: bool = True) -> plt.Figure:
    subset = _default_method_pr_subset(plot_long)
    return plot_pr_methods_figure(
        subset,
        cfg,
        title=f"{cfg.title}: default independent-signal precision/recall",
        output_suffix=f"default_methods_{cfg.output_suffix}",
        method_order=["scdrs+_conditional", "scdrs_marginal", "scdrs_marginal_leiden"],
        method_name_map={
            "scdrs+_conditional": "scDRS-FM",
            "scdrs_marginal": "scDRS",
            "scdrs_marginal_leiden": "scDRS w/ Leiden",
        },
        save=save,
    )


def build_pr_method_comparison(
    comparison: MethodComparisonConfig,
    pr_results_by_key: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    parts: list[pd.DataFrame] = []
    for member in comparison.members:
        df = pr_results_by_key.get(member.cfg_key, pd.DataFrame())
        if df.empty or "method" not in df.columns:
            continue
        sub = df[df["method"] == member.source_method].copy()
        default_signal_col = choose_default_signal_col(sub["signal_col"].dropna().unique()) if not sub.empty else None
        if default_signal_col is not None:
            sub = sub[sub["signal_col"] == default_signal_col].copy()
        if sub.empty:
            continue
        sub["method"] = member.plot_method
        sub["method_label"] = member.label
        sub["comparison"] = comparison.key
        parts.append(sub)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


In [17]:
signal_pr_results: dict[str, pd.DataFrame] = {}

for cfg in SIMULATION_SETS:
    print(f"Computing signal precision/recall: {cfg.title}")
    signal_pr_results[cfg.key] = compute_signal_precision_recall_for_set(
        cfg,
        independent_results[cfg.key],
        obs_names=obs_names,
        cell_to_leiden=cell_to_leiden,
        total_leiden_counts=total_leiden_counts,
        cell_to_scdrs_leiden=cell_to_scdrs_leiden,
        total_scdrs_leiden_counts=total_scdrs_leiden_counts,
        include_scdrs_baselines=True,
    )
    display(signal_pr_results[cfg.key].head())
    if signal_pr_results[cfg.key].empty:
        print(f"Skipping {cfg.title}: no precision/recall rows were evaluated.")
        continue
    plot_signal_variant_precision_recall_figure(signal_pr_results[cfg.key], cfg)
    plt.show()
    plot_default_method_precision_recall_figure(signal_pr_results[cfg.key], cfg)
    plt.show()

# Auxiliary sets are used for method comparisons only, so they do not include the scDRS baselines.
for cfg in AUXILIARY_SIMULATION_SETS:
    print(f"Computing signal precision/recall for comparison-only set: {cfg.title}")
    signal_pr_results[cfg.key] = compute_signal_precision_recall_for_set(
        cfg,
        independent_results[cfg.key],
        obs_names=obs_names,
        cell_to_leiden=cell_to_leiden,
        total_leiden_counts=total_leiden_counts,
        include_scdrs_baselines=False,
    )

signal_pr_comparison_results: dict[str, pd.DataFrame] = {}
for comparison in METHOD_COMPARISONS:
    base_cfg = SIM_BY_KEY[comparison.base_cfg_key]
    print(f"Plotting precision/recall comparison: {comparison.title}")
    comp_df = build_pr_method_comparison(comparison, signal_pr_results)
    signal_pr_comparison_results[comparison.key] = comp_df
    if comp_df.empty:
        print(f"Skipping {comparison.title}: no precision/recall comparison rows were available.")
        continue
    display(comp_df.head())
    plot_pr_methods_figure(
        comp_df,
        base_cfg,
        title=f"{comparison.title}: independent-signal precision/recall",
        output_suffix=f"{comparison.output_suffix}_independent_signal_pr",
        method_order=comparison.method_order,
        method_name_map=comparison.method_name_map,
    )
    plt.show()

# Backward-compatible aliases used by the original notebook.
plot_long_all = signal_pr_results["original"].copy()
plot_long = plot_long_all.copy()
prec_agg_all = _agg_mean_ci_by_signal(
    plot_long_all[plot_long_all["method"] == "scdrs+_conditional"],
    "signal_col",
    SIM_BY_KEY["original"].indep_x_col,
    "precision",
) if not plot_long_all.empty else pd.DataFrame()
rec_agg_all = _agg_mean_ci_by_signal(
    plot_long_all[plot_long_all["method"] == "scdrs+_conditional"],
    "signal_col",
    SIM_BY_KEY["original"].indep_x_col,
    "recall",
) if not plot_long_all.empty else pd.DataFrame()
prec_agg = prec_agg_all.copy()
rec_agg = rec_agg_all.copy()


Computing signal precision/recall: Original simulations


,simulation_set,cluster,replicate,src_n,causal_clusters,overlap_k,causal_genes_per_cluster,geneset_file,geneset_path,causal_leiden_labels,...,n_true_causal_clusters,precision,recall,n_inferred_populations,precision_credit_sum,n_recalled_causal_populations,marginal_file,n_sig_cells,scdrs_leiden_resolution,scdrs_leiden_min_frac_sig
0,original,0,0,1,1,50,50,TMS_FACS_0_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,0,...,1.0,1.000000,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN
1,original,0,1,1,1,50,50,TMS_FACS_0_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,0,...,1.0,1.000000,1.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN
2,original,0,2,1,1,50,50,TMS_FACS_0_2_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,0,...,1.0,0.666667,1.0,3.0,2.0,1.0,NaN,NaN,NaN,NaN
3,original,1,0,1,1,50,50,TMS_FACS_1_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,1,...,1.0,1.000000,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN
4,original,1,1,1,1,50,50,TMS_FACS_1_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,1,...,1.0,1.000000,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/all_signals_precision_recall_original.png


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/precision_recall_default_methods_original.png
Computing signal precision/recall: Cell-percent simulations


,simulation_set,cluster,replicate,src_n,causal_clusters,overlap_k,causal_genes_per_cluster,min_cell_percent,geneset_file,geneset_path,...,n_true_causal_clusters,precision,recall,n_inferred_populations,precision_credit_sum,n_recalled_causal_populations,marginal_file,n_sig_cells,scdrs_leiden_resolution,scdrs_leiden_min_frac_sig
0,sims_percent,0,0,3,3,50,50,1,TMS_FACS_0_0_pct1_ov50_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,...,3.0,0.750000,1.0,4.0,3.0,3.0,NaN,NaN,NaN,NaN
1,sims_percent,0,1,3,3,50,50,1,TMS_FACS_0_1_pct1_ov50_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,...,3.0,1.000000,1.0,3.0,3.0,3.0,NaN,NaN,NaN,NaN
2,sims_percent,0,2,3,3,50,50,1,TMS_FACS_0_2_pct1_ov50_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,...,3.0,0.833333,1.0,3.0,2.5,3.0,NaN,NaN,NaN,NaN
3,sims_percent,1,0,3,3,50,50,1,TMS_FACS_1_0_pct1_ov50_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,...,3.0,0.875000,1.0,4.0,3.5,3.0,NaN,NaN,NaN,NaN
4,sims_percent,1,1,3,3,50,50,1,TMS_FACS_1_1_pct1_ov50_src3.gs,/workspace/scdrsfm_local/results/sim/simulatio...,...,3.0,0.666667,1.0,6.0,4.0,3.0,NaN,NaN,NaN,NaN


...

Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/all_signals_precision_recall_sims_genes.png


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/precision_recall_default_methods_sims_genes.png
Computing signal precision/recall for comparison-only set: Original simulations — KNN denoising


Plotting precision/recall comparison: Denoising comparison: MAGIC vs KNN


,simulation_set,cluster,replicate,src_n,causal_clusters,overlap_k,causal_genes_per_cluster,geneset_file,geneset_path,causal_leiden_labels,...,recall,n_inferred_populations,precision_credit_sum,n_recalled_causal_populations,marginal_file,n_sig_cells,scdrs_leiden_resolution,scdrs_leiden_min_frac_sig,method_label,comparison
0,original,0,0,1,1,50,50,TMS_FACS_0_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,0,...,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,MAGIC,magic_vs_knn
1,original,0,1,1,1,50,50,TMS_FACS_0_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,0,...,1.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,MAGIC,magic_vs_knn
2,original,0,2,1,1,50,50,TMS_FACS_0_2_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,0,...,1.0,3.0,2.0,1.0,NaN,NaN,NaN,NaN,MAGIC,magic_vs_knn
3,original,1,0,1,1,50,50,TMS_FACS_1_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,1,...,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,MAGIC,magic_vs_knn
4,original,1,1,1,1,50,50,TMS_FACS_1_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,1,...,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,MAGIC,magic_vs_knn


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/precision_recall_magic_vs_knn_independent_signal_pr.png


In [18]:
from pathlib import Path

FIGURE_OUTPUT_DIR = Path("figures/signal_precision_recall")


def _move_legend_outside_axes(
    fig,
    *,
    axes_right: float = 0.78,
    legend_x: float = 0.80,
) -> None:
    """Place a consolidated legend to the right of the plotting area."""
    handles: list = []
    labels: list[str] = []

    # Collect legend entries from every axis and remove existing legends.
    for ax in fig.axes:
        ax_handles, ax_labels = ax.get_legend_handles_labels()
        handles.extend(ax_handles)
        labels.extend(ax_labels)

        legend = ax.get_legend()
        if legend is not None:
            legend.remove()

    # Preserve entries from any existing figure-level legends as well.
    for legend in list(fig.legends):
        legend_handles = getattr(legend, "legend_handles", None)
        if legend_handles is None:
            legend_handles = getattr(legend, "legendHandles", [])

        handles.extend(legend_handles)
        labels.extend(text.get_text() for text in legend.get_texts())
        legend.remove()

    # Deduplicate entries while preserving their original order.
    unique_entries: dict[str, object] = {}
    for handle, label in zip(handles, labels):
        if label and not label.startswith("_") and label not in unique_entries:
            unique_entries[label] = handle

    if not unique_entries:
        return

    # Reserve space within the figure, but outside the plotting axes.
    fig.subplots_adjust(right=axes_right)
    fig.legend(
        handles=list(unique_entries.values()),
        labels=list(unique_entries.keys()),
        loc="center left",
        bbox_to_anchor=(legend_x, 0.5),
        bbox_transform=fig.transFigure,
        frameon=False,
        borderaxespad=0,
    )


def _save_and_show_with_external_legend(
    filename_stem: str,
    *,
    output_dir: str | Path = FIGURE_OUTPUT_DIR,
    formats: tuple[str, ...] = ("png", "pdf"),
    dpi: int = 300,
) -> list[Path]:
    """Move the legend, save the current figure, and display it."""
    fig = plt.gcf()
    _move_legend_outside_axes(fig)

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    saved_paths: list[Path] = []

    for file_format in formats:
        file_format = file_format.lower().lstrip(".")
        output_path = output_dir / f"{filename_stem}.{file_format}"

        save_kwargs: dict[str, object] = {
            "bbox_inches": "tight",
        }

        # DPI is relevant for raster formats such as PNG.
        if file_format in {"png", "jpg", "jpeg", "tif", "tiff", "webp"}:
            save_kwargs["dpi"] = dpi

        fig.savefig(output_path, **save_kwargs)
        saved_paths.append(output_path)
        print(f"Saved figure: {output_path}")

    plt.show()
    return saved_paths


signal_pr_results: dict[str, pd.DataFrame] = {}

for cfg in SIMULATION_SETS:
    print(f"Computing signal precision/recall: {cfg.title}")

    signal_pr_results[cfg.key] = compute_signal_precision_recall_for_set(
        cfg,
        independent_results[cfg.key],
        obs_names=obs_names,
        cell_to_leiden=cell_to_leiden,
        total_leiden_counts=total_leiden_counts,
        cell_to_scdrs_leiden=cell_to_scdrs_leiden,
        total_scdrs_leiden_counts=total_scdrs_leiden_counts,
        include_scdrs_baselines=True,
    )

    display(signal_pr_results[cfg.key].head())

    if signal_pr_results[cfg.key].empty:
        print(f"Skipping {cfg.title}: no precision/recall rows were evaluated.")
        continue

    plot_signal_variant_precision_recall_figure(
        signal_pr_results[cfg.key],
        cfg,
    )
    _save_and_show_with_external_legend(
        f"{cfg.key}_signal_variant_precision_recall"
    )

    plot_default_method_precision_recall_figure(
        signal_pr_results[cfg.key],
        cfg,
    )
    _save_and_show_with_external_legend(
        f"{cfg.key}_default_method_precision_recall"
    )


# Auxiliary sets are used for method comparisons only, so they do not
# include the scDRS baselines.
for cfg in AUXILIARY_SIMULATION_SETS:
    print(
        "Computing signal precision/recall for comparison-only set: "
        f"{cfg.title}"
    )

    signal_pr_results[cfg.key] = compute_signal_precision_recall_for_set(
        cfg,
        independent_results[cfg.key],
        obs_names=obs_names,
        cell_to_leiden=cell_to_leiden,
        total_leiden_counts=total_leiden_counts,
        include_scdrs_baselines=False,
    )


signal_pr_comparison_results: dict[str, pd.DataFrame] = {}

for comparison in METHOD_COMPARISONS:
    base_cfg = SIM_BY_KEY[comparison.base_cfg_key]
    print(f"Plotting precision/recall comparison: {comparison.title}")

    comp_df = build_pr_method_comparison(
        comparison,
        signal_pr_results,
    )
    signal_pr_comparison_results[comparison.key] = comp_df

    if comp_df.empty:
        print(
            f"Skipping {comparison.title}: "
            "no precision/recall comparison rows were available."
        )
        continue

    display(comp_df.head())

    output_suffix = (
        f"{comparison.output_suffix}_independent_signal_pr"
    )

    plot_pr_methods_figure(
        comp_df,
        base_cfg,
        title=f"{comparison.title}: independent-signal precision/recall",
        output_suffix=output_suffix,
        method_order=comparison.method_order,
        method_name_map=comparison.method_name_map,
    )
    _save_and_show_with_external_legend(output_suffix)


# Backward-compatible aliases used by the original notebook.
plot_long_all = signal_pr_results["original"].copy()
plot_long = plot_long_all.copy()

prec_agg_all = (
    _agg_mean_ci_by_signal(
        plot_long_all[
            plot_long_all["method"] == "scdrs+_conditional"
        ],
        "signal_col",
        SIM_BY_KEY["original"].indep_x_col,
        "precision",
    )
    if not plot_long_all.empty
    else pd.DataFrame()
)

rec_agg_all = (
    _agg_mean_ci_by_signal(
        plot_long_all[
            plot_long_all["method"] == "scdrs+_conditional"
        ],
        "signal_col",
        SIM_BY_KEY["original"].indep_x_col,
        "recall",
    )
    if not plot_long_all.empty
    else pd.DataFrame()
)

prec_agg = prec_agg_all.copy()
rec_agg = rec_agg_all.copy()

Computing signal precision/recall: Original simulations


,simulation_set,cluster,replicate,src_n,causal_clusters,overlap_k,causal_genes_per_cluster,geneset_file,geneset_path,causal_leiden_labels,...,n_true_causal_clusters,precision,recall,n_inferred_populations,precision_credit_sum,n_recalled_causal_populations,marginal_file,n_sig_cells,scdrs_leiden_resolution,scdrs_leiden_min_frac_sig
0,original,0,0,1,1,50,50,TMS_FACS_0_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,0,...,1.0,1.000000,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN
1,original,0,1,1,1,50,50,TMS_FACS_0_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,0,...,1.0,1.000000,1.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN
2,original,0,2,1,1,50,50,TMS_FACS_0_2_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,0,...,1.0,0.666667,1.0,3.0,2.0,1.0,NaN,NaN,NaN,NaN
3,original,1,0,1,1,50,50,TMS_FACS_1_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,1,...,1.0,1.000000,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN
4,original,1,1,1,1,50,50,TMS_FACS_1_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,1,...,1.0,1.000000,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/all_signals_precision_recall_original.png


Saved figure: figures/signal_precision_recall/original_signal_variant_precision_recall.png


Saved figure: figures/signal_precision_recall/original_signal_variant_precision_recall.pdf


...

Saved figure: figures/signal_precision_recall/sims_genes_default_method_precision_recall.png
Saved figure: figures/signal_precision_recall/sims_genes_default_method_precision_recall.pdf
Computing signal precision/recall for comparison-only set: Original simulations — KNN denoising


Plotting precision/recall comparison: Denoising comparison: MAGIC vs KNN


,simulation_set,cluster,replicate,src_n,causal_clusters,overlap_k,causal_genes_per_cluster,geneset_file,geneset_path,causal_leiden_labels,...,recall,n_inferred_populations,precision_credit_sum,n_recalled_causal_populations,marginal_file,n_sig_cells,scdrs_leiden_resolution,scdrs_leiden_min_frac_sig,method_label,comparison
0,original,0,0,1,1,50,50,TMS_FACS_0_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,0,...,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,MAGIC,magic_vs_knn
1,original,0,1,1,1,50,50,TMS_FACS_0_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,0,...,1.0,2.0,2.0,1.0,NaN,NaN,NaN,NaN,MAGIC,magic_vs_knn
2,original,0,2,1,1,50,50,TMS_FACS_0_2_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,0,...,1.0,3.0,2.0,1.0,NaN,NaN,NaN,NaN,MAGIC,magic_vs_knn
3,original,1,0,1,1,50,50,TMS_FACS_1_0_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,1,...,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,MAGIC,magic_vs_knn
4,original,1,1,1,1,50,50,TMS_FACS_1_1_50_src1.gs,/workspace/scdrsfm_local/results/sim/simulatio...,1,...,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,MAGIC,magic_vs_knn


Saved -> /workspace/scdrsfm_local/results/sim/simulation_data/precision_recall_magic_vs_knn_independent_signal_pr.png


Saved figure: figures/signal_precision_recall/magic_vs_knn_independent_signal_pr.png
Saved figure: figures/signal_precision_recall/magic_vs_knn_independent_signal_pr.pdf


In [19]:
METHOD_COMPARISONS

[MethodComparisonConfig(key='magic_vs_knn', title='Denoising comparison: MAGIC vs KNN', base_cfg_key='original', output_suffix='magic_vs_knn', members=(MethodComparisonMember(cfg_key='original', source_method='scdrs+_conditional', plot_method='magic', label='MAGIC'), MethodComparisonMember(cfg_key='original_knn', source_method='scdrs+_conditional', plot_method='knn', label='KNN')))]

## Save analysis tables

The figures above are saved as PNGs. This final cell also writes the evaluated tables so downstream scripts can reuse the computed metrics without rerunning the notebook.


In [20]:
summary_rows = []
for cfg in ALL_EVALUATION_SETS:
    core_csv = BASE_DIR / f"core_evaluation_{cfg.output_suffix}.csv"
    indep_csv = BASE_DIR / f"independent_signal_evaluation_{cfg.output_suffix}.csv"
    pr_csv = BASE_DIR / f"signal_precision_recall_{cfg.output_suffix}.csv"

    core_results.get(cfg.key, pd.DataFrame()).to_csv(core_csv, index=False)
    independent_results.get(cfg.key, pd.DataFrame()).to_csv(indep_csv, index=False)
    signal_pr_results.get(cfg.key, pd.DataFrame()).to_csv(pr_csv, index=False)

    summary_rows.append({
        "simulation_set": cfg.key,
        "title": cfg.title,
        "core_rows": len(core_results.get(cfg.key, pd.DataFrame())),
        "independent_signal_rows": len(independent_results.get(cfg.key, pd.DataFrame())),
        "precision_recall_rows": len(signal_pr_results.get(cfg.key, pd.DataFrame())),
        "core_csv": str(core_csv),
        "independent_signal_csv": str(indep_csv),
        "precision_recall_csv": str(pr_csv),
    })

for comparison in METHOD_COMPARISONS:
    core_comp_csv = BASE_DIR / f"core_evaluation_{comparison.output_suffix}.csv"
    indep_comp_csv = BASE_DIR / f"independent_signal_evaluation_{comparison.output_suffix}.csv"
    pr_comp_csv = BASE_DIR / f"signal_precision_recall_{comparison.output_suffix}.csv"

    core_comparison_results.get(comparison.key, pd.DataFrame()).to_csv(core_comp_csv, index=False)
    independent_comparison_results.get(comparison.key, pd.DataFrame()).to_csv(indep_comp_csv, index=False)
    signal_pr_comparison_results.get(comparison.key, pd.DataFrame()).to_csv(pr_comp_csv, index=False)

    summary_rows.append({
        "simulation_set": comparison.key,
        "title": comparison.title,
        "core_rows": len(core_comparison_results.get(comparison.key, pd.DataFrame())),
        "independent_signal_rows": len(independent_comparison_results.get(comparison.key, pd.DataFrame())),
        "precision_recall_rows": len(signal_pr_comparison_results.get(comparison.key, pd.DataFrame())),
        "core_csv": str(core_comp_csv),
        "independent_signal_csv": str(indep_comp_csv),
        "precision_recall_csv": str(pr_comp_csv),
    })

analysis_summary = pd.DataFrame(summary_rows)
analysis_summary


,simulation_set,title,core_rows,independent_signal_rows,precision_recall_rows,core_csv,independent_signal_csv,precision_recall_csv
0,original,Original simulations,525,75,150,/workspace/scdrsfm_local/results/sim/simulatio...,/workspace/scdrsfm_local/results/sim/simulatio...,/workspace/scdrsfm_local/results/sim/simulatio...
1,sims_percent,Cell-percent simulations,1491,213,426,/workspace/scdrsfm_local/results/sim/simulatio...,/workspace/scdrsfm_local/results/sim/simulatio...,/workspace/scdrsfm_local/results/sim/simulatio...
2,sims_genes,Causal-gene simulations,420,60,420,/workspace/scdrsfm_local/results/sim/simulatio...,/workspace/scdrsfm_local/results/sim/simulatio...,/workspace/scdrsfm_local/results/sim/simulatio...
3,original_knn,Original simulations — KNN denoising,525,75,450,/workspace/scdrsfm_local/results/sim/simulatio...,/workspace/scdrsfm_local/results/sim/simulatio...,/workspace/scdrsfm_local/results/sim/simulatio...
4,magic_vs_knn,Denoising comparison: MAGIC vs KNN,150,150,150,/workspace/scdrsfm_local/results/sim/simulatio...,/workspace/scdrsfm_local/results/sim/simulatio...,/workspace/scdrsfm_local/results/sim/simulatio...
